### Environment install strategy

Keeps OpenVINO and NNCF installation until the deployment/export stage to avoid dependency conflicts before training.

In [1]:
# OpenVINO/NNCF are installed later only when the export/INT8 cells need them.
# Keeping installs out of the first cell avoids unnecessary dependency conflicts before training.
# Run the export cells after training; they contain their own safe install checks.


### Core dependency setup

Imports and verifies the main Python libraries needed for training, metrics, plotting, video reading, and PyTorch execution.

In [2]:
# ---- Install/confirm dependencies (Kaggle usually has these preinstalled) ----
import sys, subprocess

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

# Uncomment if a package is missing in the Kaggle environment
# pip_install("opencv-python-headless")
# pip_install("scikit-learn")
# pip_install("openvino")   # for the export section
# pip_install("tqdm")

import os
import glob
import random
import time
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast   # mixed-precision training
# fix: torch.cuda.amp.{GradScaler,autocast} are deprecated as of PyTorch 2.4 and will be
# fully removed; torch.amp.{GradScaler,autocast} is the current API. The new API takes an
# explicit device-type string ("cuda" or "cpu") as the first argument everywhere it is used
# below, instead of inferring CUDA implicitly.
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe on Kaggle)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve,
)
import pandas as pd
from tqdm.auto import tqdm

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
AMP_ENABLED = torch.cuda.is_available()   # AMP only meaningful on CUDA
# fix: torch.amp.autocast/GradScaler need an explicit device-type string. We always pass
# "cuda" when AMP is enabled (the only case that matters, since AMP_ENABLED is False on CPU);
# autocast/GradScaler simply no-op when enabled=False regardless of the device string given.
AMP_DEVICE_TYPE = "cuda"
print("Mixed-precision AMP:", AMP_ENABLED)

# fix: cap OpenCV's internal thread pool in the main process too (worker processes are
# handled separately via seed_worker). Without this, cv2 competes with PyTorch/NumPy for
# CPU threads during the main-process motion-score pass and video reads.
cv2.setNumThreads(0)

Torch: 2.10.0+cu128 | CUDA available: True
Using device: cuda
Mixed-precision AMP: True


### Dataset-path check

Finds and confirms the Kaggle dataset paths before the manifest is built.

In [3]:
# ---- Resolve the dataset root: attach your UCF-Crime subset via 'Add Input' first ----
# (Kaggle Notebook editor -> right panel -> "Add Input" -> search your dataset -> Add)

# fix: pointed at the user's actual attached dataset path and real class-folder name
# (this mirror of UCF-Crime uses "Normal_Videos_for_Event_Recognition", not "Normal").
MANUAL_DATA_ROOT = "/kaggle/input/datasets/alirakhmaev/ucf-crime-full"

EXPECTED_CLASS_NAMES = {
    "Normal_Videos_for_Event_Recognition", "Burglary", "Fighting", "Stealing"
}
KAGGLE_INPUT_ROOT = "/kaggle/input"


def autodetect_data_root(search_root, expected_classes, max_depth=3):
    """
    Walks attached Kaggle datasets looking for a directory whose immediate children
    include at least 3 of the 4 expected class-folder names.  Returns the first match,
    or None if nothing matches (caller surfaces a clear error rather than guessing wrong).
    """
    if not os.path.isdir(search_root):
        return None
    search_root_path = Path(search_root)
    for root, dirs, _files in os.walk(search_root_path):
        depth = len(Path(root).relative_to(search_root_path).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        if len(set(dirs) & expected_classes) >= 3:
            return root
    return None


if MANUAL_DATA_ROOT is not None:
    resolved_data_root = MANUAL_DATA_ROOT
    print(f"Using manually specified DATA_ROOT: {resolved_data_root}")
else:
    print(f"Scanning {KAGGLE_INPUT_ROOT} for class folders {sorted(EXPECTED_CLASS_NAMES)} ...")
    if os.path.isdir(KAGGLE_INPUT_ROOT):
        print("Attached datasets found:", os.listdir(KAGGLE_INPUT_ROOT))
    else:
        print(f"[WARN] {KAGGLE_INPUT_ROOT} does not exist — are you running this on Kaggle?")
    resolved_data_root = autodetect_data_root(KAGGLE_INPUT_ROOT, EXPECTED_CLASS_NAMES)

    if resolved_data_root is None:
        print(
            "\n[ACTION NEEDED] Could not auto-detect a folder with class subdirectories "
            f"{sorted(EXPECTED_CLASS_NAMES)} under {KAGGLE_INPUT_ROOT}.\n"
            "  1. Attach your UCF-Crime subset dataset via 'Add Input' and re-run, OR\n"
            "  2. Set MANUAL_DATA_ROOT above to the exact path and re-run.\n"
        )
    else:
        print(f"Auto-detected DATA_ROOT: {resolved_data_root}")

Using manually specified DATA_ROOT: /kaggle/input/datasets/alirakhmaev/ucf-crime-full


### Global configuration

Defines the main experiment settings: dataset paths, image size, frames per clip, training fractions, focal-loss settings, output folder, and runtime options.

In [4]:
# ---- Global configuration (mirrors the design-doc defaults) ----
class CFG:
    # --- Paths ---
    DATA_ROOT   = resolved_data_root
    MANIFEST_CSV = None   # optional CSV with columns [video_path, label]
    OUTPUT_DIR  = "/kaggle/working/checkpoints"

    # --- Class definition (binary) ---
    SUSPICIOUS_CLASSES = {"Burglary", "Fighting", "Stealing"}
    # fix: real folder name on this dataset mirror is "Normal_Videos_for_Event_Recognition",
    # not "Normal" — discover_videos() looks up CFG.DATA_ROOT/<class_name>, so this must match
    # the actual directory name exactly or zero Normal videos will be found.
    NORMAL_CLASSES     = {"Normal_Videos_for_Event_Recognition"}

    # --- Clip sampling ---
    IMG_SIZE     = 160   # spec default: 160×160
    NUM_FRAMES   = 8     # spec default sequence length
    INFER_STRIDE = 4     # sliding-window stride at inference (kept here for reference)

    # --- Splits ---
    TRAIN_FRAC = 0.70
    VAL_FRAC   = 0.15
    TEST_FRAC  = 0.15
    SEED       = 42

    # --- Training ---
    BATCH_SIZE           = 16    # spec range 8-16
    TRAIN_SAMPLES_PER_VIDEO = 2     # clip-sampling fix: draw ~2 random clips per training video per epoch
    EPOCHS               = 50    # spec range 30-50 with early stopping
    LR                   = 1e-4
    WEIGHT_DECAY         = 1e-4
    EARLY_STOP_PATIENCE  = 8     # epochs without PR-AUC improvement
    NUM_WORKERS          = 4

    # --- Loss (Focal Loss, preferred per spec) ---
    FOCAL_ALPHA = 0.75   # upweight minority (suspicious) class
    FOCAL_GAMMA = 2.5

    # --- Hard negative mining ---
    HARD_NEGATIVE_TOPK_FRAC = 0.15

    # --- Normalization (ImageNet stats, per spec) ---
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    # --- Checkpointing ---
    SAVE_LATEST    = "latest.pt"
    SAVE_BEST_PRAUC  = "best_prauc.pt"
    SAVE_BEST_ROCAUC = "best_rocauc.pt"

    # --- Motion-score cache (persisted between runs to skip re-computation) ---
    MOTION_CACHE_PATH = "/kaggle/working/motion_cache.json"

    # --- Runtime control ---
    # Keep full 3-fold CV off by default because it retrains 3 extra models and can exceed a 10-hour GPU session.
    RUN_3FOLD_CV = True
    CV_EPOCHS = 5

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)

if CFG.DATA_ROOT is None and CFG.MANIFEST_CSV is None:
    raise RuntimeError(
        "CFG.DATA_ROOT is None and no MANIFEST_CSV was set.  Go back to the auto-detection "
        "cell, attach your dataset (or set MANUAL_DATA_ROOT / CFG.MANIFEST_CSV), and re-run."
    )
print("CFG.DATA_ROOT =", CFG.DATA_ROOT)


CFG.DATA_ROOT = /kaggle/input/datasets/alirakhmaev/ucf-crime-full


### Expanded Normal manifest creation

Builds the 500-video manifest by combining all Suspicious videos with 250 Normal videos while avoiding Testing_Normal leakage.

In [5]:
from pathlib import Path
import random
import pandas as pd
import os

SEED = 42
random.seed(SEED)

# Old/current dataset: contains Burglary, Fighting, Stealing, and the old 50 Normal videos
CURRENT_ROOT = Path("/kaggle/input/datasets/alirakhmaev/ucf-crime-full")

# New full UCF-Crime dataset: use this only for extra Normal training videos
FULL_VIDEOS_ROOT = Path("/kaggle/input/datasets/vigneshwar472/ucaucf-crime-annotation-dataset/UCF_Crimes/UCF_Crimes/Videos")

# Suspicious folders from the old/current dataset
BURGLARY_DIR = CURRENT_ROOT / "Burglary"
FIGHTING_DIR = CURRENT_ROOT / "Fighting"
STEALING_DIR = CURRENT_ROOT / "Stealing"

# Normal folders
CURRENT_NORMAL_DIR = CURRENT_ROOT / "Normal_Videos_for_Event_Recognition"
FULL_NORMAL_DIR = FULL_VIDEOS_ROOT / "Training_Normal_Videos_Anomaly"

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

TARGET_NORMAL_TOTAL = 250


def list_videos(folder):
    folder = Path(folder)

    if not folder.exists():
        print("Folder does not exist:", folder)
        return []

    videos = sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS
    ])

    return videos


records = []

# 1. Add all Suspicious videos from the old/current dataset
for class_name, folder in [
    ("Burglary", BURGLARY_DIR),
    ("Fighting", FIGHTING_DIR),
    ("Stealing", STEALING_DIR),
]:
    videos = list_videos(folder)
    print(class_name, len(videos), "videos")

    if len(videos) == 0:
        raise RuntimeError(
            f"No videos found for {class_name} at {folder}. "
            "Check that the old/current UCF-Crime dataset is attached."
        )

    for p in videos:
        records.append({
            "video_path": str(p),
            "class_name": class_name,
            "label": 1
        })


# 2. Add the current 50 Normal videos
current_normal_videos = list_videos(CURRENT_NORMAL_DIR)
print("Current Normal", len(current_normal_videos), "videos")

if len(current_normal_videos) == 0:
    raise RuntimeError(
        f"No current Normal videos found at {CURRENT_NORMAL_DIR}. "
        "Check that the old/current UCF-Crime dataset is attached."
    )

for p in current_normal_videos:
    records.append({
        "video_path": str(p),
        "class_name": "Normal",
        "label": 0
    })


# 3. Add extra Normal videos from the 800 training Normal pool
full_normal_videos = list_videos(FULL_NORMAL_DIR)
print("Full training Normal available:", len(full_normal_videos), "videos")

if len(full_normal_videos) == 0:
    raise RuntimeError(
        f"No extra Normal videos found at {FULL_NORMAL_DIR}. "
        "Check that the full UCF-Crime dataset is attached."
    )

# Avoid duplicate filenames already in the current 50 Normal videos
current_normal_names = {p.name for p in current_normal_videos}

extra_normal_videos = [
    p for p in full_normal_videos
    if p.name not in current_normal_names
]

random.shuffle(extra_normal_videos)

needed_extra = TARGET_NORMAL_TOTAL - len(current_normal_videos)
selected_extra_normals = extra_normal_videos[:needed_extra]

print("Extra Normal selected:", len(selected_extra_normals), "videos")

for p in selected_extra_normals:
    records.append({
        "video_path": str(p),
        "class_name": "Normal",
        "label": 0
    })


# 4. Save manifest
manifest_df = pd.DataFrame(records)

print("\nFinal class count:")
print(manifest_df["class_name"].value_counts())

print("\nFinal binary label count:")
print(manifest_df["label"].value_counts())

# Data-problem guard: do not leak official Testing_Normal videos into training/validation/test.
# We only use Training_Normal_Videos_Anomaly for extra Normal examples.
leaked_testing_normals = manifest_df["video_path"].str.contains(
    "Testing_Normal_Videos_Anomaly", regex=False
).sum()
assert leaked_testing_normals == 0, (
    f"DATA LEAKAGE: {leaked_testing_normals} Testing_Normal_Videos_Anomaly files entered the manifest."
)
print("✓ Data audit passed — no Testing_Normal_Videos_Anomaly leakage in manifest.")

MANIFEST_PATH = "/kaggle/working/expanded_normal_manifest.csv"
manifest_df.to_csv(MANIFEST_PATH, index=False)

print("\nSaved manifest to:", MANIFEST_PATH)


# 5. Tell the main notebook to use this manifest
CFG.MANIFEST_CSV = MANIFEST_PATH

# Use a fresh output folder for this MobileNetV3-Large + TSM run.
# This prevents accidental resume from an older compatible checkpoint trained with earlier notebook logic.
CFG.OUTPUT_DIR = "/kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed"
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

print("\nCFG.MANIFEST_CSV =", CFG.MANIFEST_CSV)
print("CFG.OUTPUT_DIR   =", CFG.OUTPUT_DIR)


Burglary 100 videos
Fighting 50 videos
Stealing 100 videos
Current Normal 50 videos
Full training Normal available: 800 videos
Extra Normal selected: 200 videos

Final class count:
class_name
Normal      250
Burglary    100
Stealing    100
Fighting     50
Name: count, dtype: int64

Final binary label count:
label
1    250
0    250
Name: count, dtype: int64
✓ Data audit passed — no Testing_Normal_Videos_Anomaly leakage in manifest.

Saved manifest to: /kaggle/working/expanded_normal_manifest.csv

CFG.MANIFEST_CSV = /kaggle/working/expanded_normal_manifest.csv
CFG.OUTPUT_DIR   = /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed


### Manifest loading and class-balanced split

Loads video paths, builds records, and creates video-level train/validation/test splits balanced by class_name with leakage checks.

In [6]:
def discover_videos(data_root, suspicious_classes, normal_classes):
    """
    Expects a folder layout like:
        data_root/
            Normal/*.mp4
            Burglary/*.mp4
            Fighting/*.mp4
            Stealing/*.mp4
    Returns list of dicts: {path, class_name, label}
    """
    records = []
    all_classes = suspicious_classes | normal_classes
    for cls in all_classes:
        cls_dir = os.path.join(data_root, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] class dir not found: {cls_dir}")
            continue
        video_paths = (
            sorted(glob.glob(os.path.join(cls_dir, "**", "*.mp4"), recursive=True))
            + sorted(glob.glob(os.path.join(cls_dir, "**", "*.avi"), recursive=True))
        )
        label = 1 if cls in suspicious_classes else 0
        for vp in video_paths:
            records.append({"path": vp, "class_name": cls, "label": label})
    return records


def load_manifest_from_csv(csv_path, suspicious_classes):
    df = pd.read_csv(csv_path)

    if "video_path" not in df.columns:
        raise ValueError(
            f"Manifest CSV must contain a 'video_path' column. Found columns: {list(df.columns)}"
        )
    if "label" not in df.columns and "class_name" not in df.columns:
        raise ValueError(
            "Manifest CSV must contain either 'label' or 'class_name' so binary labels can be assigned."
        )

    records = []
    for _, row in df.iterrows():
        cls = row.get("class_name", None)
        if pd.isna(cls):
            cls = None

        if "label" in df.columns and not pd.isna(row["label"]):
            label = int(row["label"])
        else:
            label = 1 if cls in suspicious_classes else 0

        records.append({"path": row["video_path"], "class_name": cls, "label": int(label)})
    return records


def build_manifest():
    if CFG.MANIFEST_CSV is not None and os.path.exists(CFG.MANIFEST_CSV):
        records = load_manifest_from_csv(CFG.MANIFEST_CSV, CFG.SUSPICIOUS_CLASSES)
    else:
        records = discover_videos(CFG.DATA_ROOT, CFG.SUSPICIOUS_CLASSES, CFG.NORMAL_CLASSES)
    if len(records) == 0:
        raise RuntimeError(
            "No videos found.  Check CFG.DATA_ROOT / CFG.MANIFEST_CSV and dataset attachment."
        )
    return records


def stratified_video_split(records, train_frac, val_frac, test_frac, seed):
    """
    VIDEO-LEVEL stratified split.

    This version stratifies by class_name, not only by binary label.
    The model is still binary (0=Normal, 1=Suspicious), but this keeps
    Normal, Burglary, Fighting, and Stealing more evenly distributed across
    train/validation/test.
    """
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6

    class_labels = [r["class_name"] for r in records]

    train_recs, rest_recs = train_test_split(
        records,
        train_size=train_frac,
        stratify=class_labels,
        random_state=seed,
    )

    rest_class_labels = [r["class_name"] for r in rest_recs]
    val_size_within_rest = val_frac / (val_frac + test_frac)

    val_recs, test_recs = train_test_split(
        rest_recs,
        train_size=val_size_within_rest,
        stratify=rest_class_labels,
        random_state=seed,
    )

    # ── Leakage audit ────────────────────────────────────────────────────────
    train_paths = {r["path"] for r in train_recs}
    val_paths   = {r["path"] for r in val_recs}
    test_paths  = {r["path"] for r in test_recs}

    train_val_overlap  = train_paths & val_paths
    train_test_overlap = train_paths & test_paths
    val_test_overlap   = val_paths & test_paths

    assert len(train_val_overlap)  == 0, f"LEAKAGE: {len(train_val_overlap)} videos in train ∩ val"
    assert len(train_test_overlap) == 0, f"LEAKAGE: {len(train_test_overlap)} videos in train ∩ test"
    assert len(val_test_overlap)   == 0, f"LEAKAGE: {len(val_test_overlap)} videos in val ∩ test"

    print("✓ Split leakage audit passed — zero video overlap across all three splits.")

    return train_recs, val_recs, test_recs


manifest = build_manifest()
print(f"Total videos discovered: {len(manifest)}")
print(Counter([r['class_name'] for r in manifest]))

train_records, val_records, test_records = stratified_video_split(
    manifest, CFG.TRAIN_FRAC, CFG.VAL_FRAC, CFG.TEST_FRAC, CFG.SEED
)
print(f"Train: {len(train_records)} | Val: {len(val_records)} | Test: {len(test_records)}")
print("Train label balance:", Counter([r['label'] for r in train_records]))
print("Val   label balance:", Counter([r['label'] for r in val_records]))
print("Test  label balance:", Counter([r['label'] for r in test_records]))

print("Train class balance:", Counter([r["class_name"] for r in train_records]))
print("Val   class balance:", Counter([r["class_name"] for r in val_records]))
print("Test  class balance:", Counter([r["class_name"] for r in test_records]))

Total videos discovered: 500
Counter({'Normal': 250, 'Burglary': 100, 'Stealing': 100, 'Fighting': 50})
✓ Split leakage audit passed — zero video overlap across all three splits.
Train: 350 | Val: 75 | Test: 75
Train label balance: Counter({1: 175, 0: 175})
Val   label balance: Counter({0: 38, 1: 37})
Test  label balance: Counter({1: 38, 0: 37})
Train class balance: Counter({'Normal': 175, 'Stealing': 70, 'Burglary': 70, 'Fighting': 35})
Val   class balance: Counter({'Normal': 38, 'Stealing': 15, 'Burglary': 15, 'Fighting': 7})
Test  class balance: Counter({'Normal': 37, 'Stealing': 15, 'Burglary': 15, 'Fighting': 8})


### Video frame reader

Reads frames from each video clip efficiently, supporting random, center, and sliding-window sampling modes.

In [7]:
def read_frames_with_count(path, num_frames, img_size, mode, stride=4, start=0):
    """
    Single-pass version: opens the video once, reads the frame count off the same handle,
    decides sampling indices, then decodes only the needed frames.
    """
    cap   = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(total, num_frames, mode=mode, stride=stride, start=start)

    if total <= 0:
        cap.release()
        return [np.zeros((img_size, img_size, 3), dtype=np.uint8) for _ in indices], indices

    idx_set    = sorted(set(indices))
    idx_to_frame = {}
    max_idx    = max(idx_set)

    # Runtime fix: jump close to the first needed frame instead of decoding from frame 0.
    # This helps both random training clips and sliding-window evaluation, especially on long videos.
    if len(idx_set) > 0:
        first_idx = idx_set[0]
        cap.set(cv2.CAP_PROP_POS_FRAMES, first_idx)
        pos = first_idx
    else:
        pos = 0

    while pos <= max_idx:
        ret, frame = cap.read()
        if not ret:
            break
        if pos in idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (img_size, img_size), interpolation=cv2.INTER_AREA)
            idx_to_frame[pos] = frame
        pos += 1
    cap.release()

    if len(idx_to_frame) == 0:
        return [np.zeros((img_size, img_size, 3), dtype=np.uint8) for _ in indices], indices

    last_good = next(iter(idx_to_frame.values()))
    out = []
    for i in indices:
        if i in idx_to_frame:
            last_good = idx_to_frame[i]
        out.append(last_good)
    return out, indices


def sample_frame_indices(total_frames, num_frames, mode="random", stride=4, start=0):
    """
    mode='random'  -> training: random window + jitter
    mode='uniform' -> val/test: deterministic, evenly spaced
    mode='window'  -> deterministic sliding window at inference
    """
    if total_frames <= 0:
        return [0] * num_frames

    if mode == "window":
        return [min(start + i * stride, total_frames - 1) for i in range(num_frames)]

    if mode == "uniform":
        if total_frames <= num_frames:
            return list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
        return np.linspace(0, total_frames - 1, num_frames).astype(int).tolist()

    # mode == 'random' (training)
    needed_span = num_frames * stride if stride else num_frames * 4
    if total_frames <= needed_span:
        return sorted(np.random.choice(total_frames, num_frames, replace=True).tolist())
    max_start  = total_frames - needed_span
    rnd_start  = random.randint(0, max_start)
    eff_stride = stride if stride else max(1, needed_span // num_frames)
    idxs = [min(rnd_start + i * eff_stride + random.randint(-1, 1), total_frames - 1)
            for i in range(num_frames)]
    return sorted([max(0, i) for i in idxs])

### Clip preprocessing and augmentation

Applies resizing, normalization, and training-time augmentation to video clips before they enter the model.

In [8]:
class ClipAugmenter:
    """Applies the same spatial transform across every frame of a clip (train only)."""

    def __init__(self, img_size, mean, std, train=True):
        self.img_size = img_size
        self.mean  = np.array(mean, dtype=np.float32)
        self.std   = np.array(std,  dtype=np.float32)
        self.train = train

    def _get_train_params(self, h, w):
        """
        Draw ALL random parameters ONCE per clip.
        Crop offsets (top/left) are included here so every frame in the clip
        receives the same crop window — fixing the per-frame jitter bug in v1.
        """
        crop_scale = random.uniform(0.85, 1.0)
        ch = int(h * crop_scale)
        cw = int(w * crop_scale)
        top  = random.randint(0, max(h - ch, 0))
        left = random.randint(0, max(w - cw, 0))
        return dict(
            flip       = random.random() < 0.5,
            jitter     = random.random() < 0.5,
            blur       = random.random() < 0.2,
            brightness = random.uniform(0.8, 1.2),
            contrast   = random.uniform(0.8, 1.2),
            saturation = random.uniform(0.8, 1.2),
            crop_scale = crop_scale,
            crop_top   = top,
            crop_left  = left,
            crop_h     = ch,
            crop_w     = cw,
        )

    def _apply_to_frame(self, frame, params):
        if params["crop_scale"] < 1.0:
            t, l = params["crop_top"], params["crop_left"]
            ch,  cw = params["crop_h"], params["crop_w"]
            frame = frame[t:t+ch, l:l+cw]
            frame = cv2.resize(frame, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)

        if params["flip"]:
            frame = cv2.flip(frame, 1)

        if params["jitter"]:
            img  = frame.astype(np.float32)
            img *= params["brightness"]
            mean_pix = img.mean(axis=(0, 1), keepdims=True)
            img  = (img - mean_pix) * params["contrast"] + mean_pix
            gray = img.mean(axis=2, keepdims=True)
            img  = (img - gray) * params["saturation"] + gray
            frame = np.clip(img, 0, 255).astype(np.uint8)

        if params["blur"]:
            frame = cv2.GaussianBlur(frame, (3, 3), 0)

        return frame

    def __call__(self, frames):
        # Draw params once using the first frame's dimensions
        h, w, _ = frames[0].shape
        params = self._get_train_params(h, w) if self.train else {}
        out = []
        for f in frames:
            if self.train:
                f = self._apply_to_frame(f, params)
            f = f.astype(np.float32) / 255.0
            f = (f - self.mean) / self.std
            out.append(f.transpose(2, 0, 1))   # HWC → CHW
        clip = np.stack(out, axis=0)            # [T, C, H, W]
        return torch.from_numpy(clip).float()

### DataLoader seeding and datasets

Sets reproducible DataLoader behavior and defines the dataset objects used to sample clips from each video.

In [9]:
def seed_worker(worker_id):
    """
    Ensures DataLoader workers produce deterministic results across runs.

    fix: also disables OpenCV's internal thread pool inside each worker process.
    Without this, every one of the `num_workers` processes spawns its own OpenCV
    thread pool on top of PyTorch's own multiprocessing workers, oversubscribing
    CPU cores and causing severe (sometimes near-hanging) slowdowns when decoding
    video with cv2.VideoCapture inside Dataset.__getitem__ — a well-known issue
    combining OpenCV + multiprocess DataLoaders, including on Kaggle.
    """
    cv2.setNumThreads(0)
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(CFG.SEED)


class UCFCrimeClipDataset(Dataset):
    def __init__(self, records, cfg, train=True, motion_scores=None):
        self.records     = records
        self.cfg         = cfg
        self.train       = train
        self.augmenter   = ClipAugmenter(cfg.IMG_SIZE, cfg.MEAN, cfg.STD, train=train)
        self.motion_scores = motion_scores or {}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec          = self.records[idx]
        path, label  = rec["path"], rec["label"]
        mode         = "random" if self.train else "uniform"
        frames, _idx = read_frames_with_count(
            path, self.cfg.NUM_FRAMES, self.cfg.IMG_SIZE,
            mode=mode, stride=self.cfg.INFER_STRIDE,
        )
        clip = self.augmenter(frames)   # [T, C, H, W]
        return clip, torch.tensor(label, dtype=torch.float32), path


# ── Background-subtraction hard-negative mining ─────────────────────────────
def estimate_motion_score_mog2(path, num_probe_frames=10):
    """
    MOG2 background subtractor — more robust than frame-differencing.
    Frame differencing conflates camera shake, lighting flicker, and rain with
    genuine foreground motion.  MOG2 explicitly models the background distribution
    and outputs a foreground mask, making it less sensitive to those artefacts.

    Returns mean foreground pixel fraction across sampled frames (higher = more motion).
    """
    cap   = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return 0.0

    mog2   = cv2.createBackgroundSubtractorMOG2(detectShadows=False)
    idxs   = np.linspace(0, total - 1, num_probe_frames).astype(int)
    scores = []
    pos    = 0
    idx_set = set(idxs.tolist())
    while pos <= idxs[-1]:
        ret, frame = cap.read()
        if not ret:
            break
        if pos in idx_set:
            small = cv2.resize(frame, (64, 64))
            mask  = mog2.apply(small)
            scores.append(mask.mean() / 255.0)   # fraction of pixels marked foreground
        pos += 1
    cap.release()
    return float(np.mean(scores)) if scores else 0.0


def load_motion_cache(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}


def save_motion_cache(cache, path):
    with open(path, "w") as f:
        json.dump(cache, f)


def build_class_balanced_sampler(records, cfg, hard_negative_topk_frac=0.15, motion_cache=None):
    """
    Two-level balanced sampler.

    Fixes a data-imbalance issue without changing the task from binary classification:
      1) keeps the epoch roughly balanced as Normal vs Suspicious;
      2) spreads the Suspicious half across Burglary, Fighting, and Stealing so the
         smaller Fighting class is not under-sampled;
      3) still upweights high-motion Normal videos as hard negatives.
    """
    class_names = np.array([r.get("class_name", "Unknown") for r in records])
    labels      = np.array([r["label"] for r in records])

    suspicious_class_names = sorted({
        cls for cls, lab in zip(class_names, labels)
        if lab == 1 and cls != "Unknown"
    })

    weights = np.zeros(len(records), dtype=np.float64)

    # Reserve half the sampling probability mass for Normal and half for Suspicious.
    normal_idxs = np.where(labels == 0)[0]
    if len(normal_idxs) > 0:
        weights[normal_idxs] = 0.5 / len(normal_idxs)

    # Share the Suspicious half equally across the suspicious subclasses.
    if len(suspicious_class_names) > 0:
        per_suspicious_class_mass = 0.5 / len(suspicious_class_names)
        for cls in suspicious_class_names:
            cls_idxs = np.where((labels == 1) & (class_names == cls))[0]
            if len(cls_idxs) > 0:
                weights[cls_idxs] = per_suspicious_class_mass / len(cls_idxs)

    # Fallback safety if a class name is missing or weights did not fill correctly.
    if not np.isfinite(weights).all() or weights.sum() <= 0:
        labels_list = labels.tolist()
        class_counts = Counter(labels_list)
        inv_freq = {c: 1.0 / class_counts[c] for c in class_counts}
        weights = np.array([inv_freq[l] for l in labels], dtype=np.float64)

    motion_cache = motion_cache or {}

    # Hard-negative mining: only high-motion Normal videos receive an extra boost.
    if len(normal_idxs) > 0 and hard_negative_topk_frac > 0:
        scores = []
        for i in normal_idxs:
            p = records[i]["path"]
            if p not in motion_cache:
                motion_cache[p] = estimate_motion_score_mog2(p)
            scores.append(motion_cache[p])

        scores = np.array(scores)
        k = max(1, int(len(scores) * hard_negative_topk_frac))
        hard_global_idx = normal_idxs[np.argsort(scores)[-k:]]
        weights[hard_global_idx] *= 1.5   # moderate boost; avoids overwhelming rare suspicious clips

    # Normalize after hard-negative boost.
    weights = weights / weights.sum()

    samples_per_video = int(getattr(cfg, "TRAIN_SAMPLES_PER_VIDEO", 1))
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights) * max(1, samples_per_video),
        replacement=True,
    )

    # Lightweight audit so the user can see what the sampler is doing.
    print("Sampler class counts:", Counter(class_names.tolist()))
    expected_class_draws = {}
    for cls in sorted(set(class_names.tolist())):
        cls_mass = float(weights[class_names == cls].sum())
        expected_class_draws[cls] = round(cls_mass * len(weights) * max(1, samples_per_video), 1)
    print("Expected sampled clips/epoch by class:", expected_class_draws)

    return sampler, motion_cache

train_dataset = UCFCrimeClipDataset(train_records, CFG, train=True)
val_dataset   = UCFCrimeClipDataset(val_records,   CFG, train=False)
test_dataset  = UCFCrimeClipDataset(test_records,  CFG, train=False)

print("Estimating motion scores for hard-negative mining (MOG2; cached after first run)...")
motion_cache = load_motion_cache(CFG.MOTION_CACHE_PATH)
train_sampler, motion_cache = build_class_balanced_sampler(
    train_records, CFG,
    hard_negative_topk_frac=CFG.HARD_NEGATIVE_TOPK_FRAC,
    motion_cache=motion_cache,
)
save_motion_cache(motion_cache, CFG.MOTION_CACHE_PATH)
print(f"Motion cache saved to {CFG.MOTION_CACHE_PATH} ({len(motion_cache)} entries).")

# ── Fix: drop_last only on train loader; eval loaders use full dataset ───────
# drop_last=True is kept for train because replacement sampling and BatchNorm are more stable
# with full batches. Val/test can safely use drop_last=False because TSM only requires
# each sample to contain CFG.NUM_FRAMES frames; it does not require a fixed batch size.
def make_eval_loader(dataset, cfg):
    """Evaluation loader: keeps the final partial batch; no samples are discarded."""
    return DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=(cfg.NUM_WORKERS > 0),
    )

train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE, sampler=train_sampler,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
    worker_init_fn=seed_worker, generator=g,
    persistent_workers=(CFG.NUM_WORKERS > 0),
)
val_loader  = make_eval_loader(val_dataset,  CFG)
test_loader = make_eval_loader(test_dataset, CFG)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
print("Note: train uses replacement sampling; repeated video draws produce different random clips.")
print(f"Train samples/video per epoch: {getattr(CFG, 'TRAIN_SAMPLES_PER_VIDEO', 1)}")
print("Note: train uses drop_last=True (TSM reshape constraint); val/test use full dataset.")


Estimating motion scores for hard-negative mining (MOG2; cached after first run)...
Sampler class counts: Counter({'Normal': 175, 'Stealing': 70, 'Burglary': 70, 'Fighting': 35})
Expected sampled clips/epoch by class: {'Burglary': 112.5, 'Fighting': 112.5, 'Normal': 362.5, 'Stealing': 112.5}
Motion cache saved to /kaggle/working/motion_cache.json (175 entries).
Train batches: 43 | Val batches: 5 | Test batches: 5
Note: train uses replacement sampling; repeated video draws produce different random clips.
Train samples/video per epoch: 2
Note: train uses drop_last=True (TSM reshape constraint); val/test use full dataset.


### MobileNetV3-Large + TSM model

Defines the lightweight temporal model using MobileNetV3-Large with Temporal Shift Module support for video clips.

In [10]:
class TemporalShift(nn.Module):
    """
    TSM channel-shift operation.
    Input shape during shift: [N*T, C, H, W] → reshaped to [N, T, C, H, W],
    a portion of channels is shifted +1 step, another -1 step, rest unchanged.
    """
    def __init__(self, num_frames, shift_div=8):
        super().__init__()
        self.num_frames = num_frames
        self.shift_div  = shift_div

    def forward(self, x):
        nt, c, h, w = x.size()
        if nt % self.num_frames != 0:
            raise RuntimeError(
                f"TemporalShift got {nt} frame-tensors, not divisible by "
                f"num_frames={self.num_frames}.  Use drop_last=True on train loader."
            )
        n_batch = nt // self.num_frames
        x   = x.view(n_batch, self.num_frames, c, h, w)
        fold = c // self.shift_div
        out = torch.zeros_like(x)
        out[:, :-1, :fold]          = x[:, 1:, :fold]
        out[:, 1:,  fold:2*fold]    = x[:, :-1, fold:2*fold]
        out[:, :,   2*fold:]        = x[:, :,   2*fold:]
        return out.view(nt, c, h, w)


class TSMBlockWrapper(nn.Module):
    """Wraps an existing conv block so TSM is applied right before it."""
    def __init__(self, block, num_frames, shift_div=8):
        super().__init__()
        self.shift = TemporalShift(num_frames, shift_div)
        self.block = block

    def forward(self, x):
        return self.block(self.shift(x))


def insert_tsm_into_mobilenetv3(backbone, num_frames, shift_div=8, every_n_blocks=1):
    """
    Wraps every Nth inverted-residual block of MobileNetV3's feature extractor with TSM.
    every_n_blocks=1    → shift before every block (standard TSM).
    every_n_blocks=2    → lighter profile; see ablation study in Section 5a.
    every_n_blocks=None → no TSM wrapper; still processes all T frames and temporal-averages.
    """
    if every_n_blocks is None or every_n_blocks <= 0:
        return backbone

    features = backbone.features
    count = 0
    for i, layer in enumerate(features):
        if i == 0:
            continue    # skip the stem conv
        count += 1
        if count % every_n_blocks == 0:
            features[i] = TSMBlockWrapper(layer, num_frames, shift_div)
    return backbone


class SuspiciousActivityModel(nn.Module):
    """
    MobileNetV3-Large → TSM → Global Average Pooling → Dropout → Linear → Binary Logit
    (Model A from the design spec)
    """
    def __init__(self, num_frames=8, dropout=0.3, pretrained=True,
                 shift_div=8, every_n_blocks=1):
        super().__init__()
        self.num_frames = num_frames
        # Use ImageNet weights when available. On Kaggle, downloading pretrained
        # weights can fail if internet is disabled; this fallback prevents the notebook
        # from crashing and clearly tells you when training starts from random weights.
        weights = None
        if pretrained:
            try:
                weights = torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
                backbone = torchvision.models.mobilenet_v3_large(weights=weights)
            except Exception as e:
                print("[WARN] Could not load/download MobileNetV3-Large pretrained weights.")
                print("       Falling back to random initialization. Reason:", repr(e))
                backbone = torchvision.models.mobilenet_v3_large(weights=None)
        else:
            backbone = torchvision.models.mobilenet_v3_large(weights=None)

        backbone = insert_tsm_into_mobilenetv3(backbone, num_frames, shift_div, every_n_blocks)
        self.features = backbone.features
        self.gap      = nn.AdaptiveAvgPool2d(1)
        feat_dim      = backbone.classifier[0].in_features   # 960 for MobileNetV3-Large
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(feat_dim, 1)

    def forward(self, x):
        n, t, c, h, w = x.shape
        assert t == self.num_frames, f"Expected {self.num_frames} frames, got {t}"
        x     = x.view(n * t, c, h, w)
        feats = self.features(x)
        feats = self.gap(feats).flatten(1)
        feats = feats.view(n, t, -1).mean(dim=1)   # temporal avg pool → [N, C']
        feats = self.dropout(feats)
        return self.fc(feats).squeeze(1)            # [N]


model = SuspiciousActivityModel(
    num_frames=CFG.NUM_FRAMES, dropout=0.3, pretrained=True
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,} ({n_params / 1e6:.2f} M)")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 134MB/s] 


Total parameters: 2,972,913 (2.97 M)


### Focal loss

Defines focal loss for binary suspicious-vs-normal classification, helping the model focus on harder training examples.

In [11]:
class FocalLoss(nn.Module):
    """Binary focal loss.  alpha upweights the positive (suspicious) class."""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce      = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs    = torch.sigmoid(logits)
        p_t      = probs * targets + (1 - probs) * (1 - targets)
        alpha_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss     = alpha_t * (1 - p_t).pow(self.gamma) * bce
        return loss.mean()


criterion = FocalLoss(alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)
scaler = GradScaler(AMP_DEVICE_TYPE, enabled=AMP_ENABLED)   # mixed-precision gradient scaler

### Metric computation

Computes validation/test metrics such as PR-AUC, ROC-AUC, recall, precision, F1, MSE, and RMSE.

In [12]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    pr_auc = average_precision_score(y_true, y_prob)
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = float("nan")

    # fix: added RMSE/MSE. Note these are computed between the raw probability y_prob
    # and the binary label y_true (0/1), NOT between y_pred and y_true -- squared error
    # against a hard 0/1 prediction collapses to the same information as accuracy and
    # is not a standard reporting choice. MSE-of-probability-vs-binary-label is formally
    # the Brier score, a real and standard metric for probabilistic binary classifiers
    # (lower is better; measures both calibration and discrimination jointly). RMSE is
    # just its square root, included only for direct comparability since it was
    # specifically requested -- it does not carry additional information beyond MSE
    # for this use case, and neither metric should be read the way RMSE/MSE are read for
    # a continuous regression target (e.g. there is no natural "units" interpretation,
    # since the target is categorical not continuous).
    mse  = float(np.mean((y_prob - y_true) ** 2))
    rmse = float(np.sqrt(mse))

    return {
        "pr_auc":    pr_auc,
        "roc_auc":   roc_auc,
        "f1":        f1_score(y_true,        y_pred, zero_division=0),
        "recall":    recall_score(y_true,    y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "mse":       mse,
        "rmse":      rmse,
    }


def best_threshold_for_f1(y_true, y_prob):
    """Grid search over thresholds maximising F1 on the validation set."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * precisions * recalls / np.clip(precisions + recalls, 1e-8, None)
    best_idx   = np.nanargmax(f1s[:-1]) if len(f1s) > 1 else 0
    best_thresh = thresholds[best_idx] if len(thresholds) > 0 else 0.5
    return float(best_thresh), float(f1s[best_idx])

### Checkpoint management

Handles saving/loading model checkpoints and protects against incompatible checkpoint resumes.

In [13]:
class CheckpointManager:
    """Tracks and saves latest / best-PR-AUC / best-ROC-AUC model states."""
    def __init__(self, output_dir, cfg):
        self.output_dir  = output_dir
        self.cfg         = cfg
        self.best_pr_auc  = -1.0
        self.best_roc_auc = -1.0
        os.makedirs(output_dir, exist_ok=True)

    def _save(self, filename, model, optimizer, epoch, metrics, scheduler=None, scaler=None):
        path = os.path.join(self.output_dir, filename)
        # fix: scheduler and scaler state are now saved alongside model/optimizer.
        # ReduceLROnPlateau tracks patience/cooldown/best internally via its own state_dict,
        # and GradScaler tracks its dynamic loss-scale factor via its own state_dict. Neither
        # was being persisted before, so resuming from a checkpoint silently reset the LR
        # scheduler's patience counter and the AMP scaler's loss scale to their initial values
        # instead of continuing from where training actually left off.
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state": scaler.state_dict() if scaler is not None else None,
            "metrics": metrics,
        }, path)
        return path

    def update(self, model, optimizer, epoch, val_metrics, scheduler=None, scaler=None):
        improved_pr = False
        self._save(self.cfg.SAVE_LATEST, model, optimizer, epoch, val_metrics, scheduler, scaler)
        if val_metrics["pr_auc"] > self.best_pr_auc:
            self.best_pr_auc = val_metrics["pr_auc"]
            self._save(self.cfg.SAVE_BEST_PRAUC, model, optimizer, epoch, val_metrics, scheduler, scaler)
            improved_pr = True
            print(f"  -> New best PR-AUC: {self.best_pr_auc:.4f}  (saved {self.cfg.SAVE_BEST_PRAUC})")
        if not math.isnan(val_metrics["roc_auc"]) and val_metrics["roc_auc"] > self.best_roc_auc:
            self.best_roc_auc = val_metrics["roc_auc"]
            self._save(self.cfg.SAVE_BEST_ROCAUC, model, optimizer, epoch, val_metrics, scheduler, scaler)
            print(f"  -> New best ROC-AUC: {self.best_roc_auc:.4f}  (saved {self.cfg.SAVE_BEST_ROCAUC})")
        return improved_pr


ckpt_manager = CheckpointManager(CFG.OUTPUT_DIR, CFG)

### One training/evaluation epoch

Runs one full pass over a loader, either training the model or evaluating it without weight updates.

In [14]:
def run_epoch(model, loader, optimizer, criterion, device, scaler, train=True, desc=None):
    model.train() if train else model.eval()
    total_loss    = 0.0
    n_samples     = 0
    all_targets, all_probs = [], []

    # fix: inference_mode is faster/lower-memory than no_grad for the eval path;
    # set_grad_enabled is still required for the train path since it needs autograd.
    grad_ctx = torch.enable_grad() if train else torch.inference_mode()
    # fix: a tqdm progress bar with live loss is added so long epochs (on-the-fly video
    # decoding can make a single epoch take many minutes) show visible progress on Kaggle
    # instead of appearing to hang silently.
    pbar = tqdm(loader, desc=desc or ("train" if train else "val"), leave=False)
    with grad_ctx:
        for clips, labels, _paths in pbar:
            clips  = clips.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)   # fix: avoids zeroing grad tensors, saves memory/time

            # Mixed-precision forward pass (no-op on CPU / if AMP disabled)
            with autocast(AMP_DEVICE_TYPE, enabled=AMP_ENABLED):
                logits = model(clips)
                loss   = criterion(logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()

            bs = clips.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
            probs = torch.sigmoid(logits).detach().cpu().float().numpy()
            all_probs.extend(probs.tolist())
            all_targets.extend(labels.detach().cpu().numpy().tolist())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / max(n_samples, 1)
    # Use tuned threshold=0.5 during training loop only for monitoring;
    # the real decision threshold is selected on the validation set after training.
    metrics = compute_metrics(all_targets, all_probs, threshold=0.5)
    metrics["loss"] = avg_loss
    return metrics

### Main training loop

Trains the model, tracks validation metrics, saves the best checkpoints, and stops early when PR-AUC no longer improves.

In [15]:
# fix: added resume-from-checkpoint support. Kaggle GPU sessions can disconnect or hit the
# 9-12h runtime cap before CFG.EPOCHS finishes; without this, re-running this cell after a
# disconnect silently started over from epoch 1 and overwrote the in-progress run. Now it
# checks for an existing latest.pt and, if found, resumes model/optimizer state, the epoch
# counter, and best-metric tracking (read independently from each best-* checkpoint, since
# the best PR-AUC and best ROC-AUC don't necessarily occur on the same epoch).
history = {"train": [], "val": []}
start_epoch = 1
epochs_without_improvement = 0

latest_ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.SAVE_LATEST)
if os.path.exists(latest_ckpt_path):
    print(f"Found existing checkpoint at {latest_ckpt_path} — attempting to resume.")
    # fix: weights_only=False — PyTorch 2.6 changed torch.load's default to weights_only=True,
    # which rejects the numpy scalars stored inside each checkpoint's "metrics" dict. Same fix
    # as the test-eval and threshold-search cells, applied here to all three resume-path loads.
    try:
        ckpt = torch.load(latest_ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])

        # fix: scheduler and scaler state are now restored too (see CheckpointManager fix above).
        # Without this, ReduceLROnPlateau's patience/cooldown counters and GradScaler's dynamic
        # loss-scale factor silently reset to their initial values on every resume, producing a
        # subtly different LR-decay schedule than an uninterrupted run would have had. The "if"
        # guards handle checkpoints saved before this fix existed (scheduler_state/scaler_state
        # keys absent or None), so old checkpoints can still be resumed without crashing.
        if ckpt.get("scheduler_state") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        if ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1

        best_prauc_path = os.path.join(CFG.OUTPUT_DIR, CFG.SAVE_BEST_PRAUC)
        if os.path.exists(best_prauc_path):
            ckpt_manager.best_pr_auc = torch.load(best_prauc_path, map_location=DEVICE, weights_only=False)["metrics"].get("pr_auc", -1.0)
        best_rocauc_path = os.path.join(CFG.OUTPUT_DIR, CFG.SAVE_BEST_ROCAUC)
        if os.path.exists(best_rocauc_path):
            ckpt_manager.best_roc_auc = torch.load(best_rocauc_path, map_location=DEVICE, weights_only=False)["metrics"].get("roc_auc", -1.0)

        history_path = os.path.join(CFG.OUTPUT_DIR, "history.json")
        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)
        print(f"Resuming from epoch {start_epoch} "
              f"(best PR-AUC so far: {ckpt_manager.best_pr_auc:.4f}, "
              f"best ROC-AUC so far: {ckpt_manager.best_roc_auc:.4f})")
    except Exception as e:
        print("[WARN] Existing checkpoint could not be loaded into this model.")
        print("       This usually happens after changing the backbone or model shape.")
        print("       Starting fresh instead. Reason:", str(e).split("\n")[0])
        start_epoch = 1
        epochs_without_improvement = 0
else:
    print("No existing checkpoint found — starting fresh from epoch 1.")

if start_epoch > CFG.EPOCHS:
    print(f"[INFO] Resumed epoch ({start_epoch}) already exceeds CFG.EPOCHS ({CFG.EPOCHS}); "
          f"nothing left to train. Raise CFG.EPOCHS to continue training further.")

for epoch in range(start_epoch, CFG.EPOCHS + 1):
    t0 = time.time()

    train_metrics = run_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler,
                               train=True, desc=f"Epoch {epoch}/{CFG.EPOCHS} [train]")
    val_metrics   = run_epoch(model, val_loader,   optimizer, criterion, DEVICE, scaler,
                               train=False, desc=f"Epoch {epoch}/{CFG.EPOCHS} [val]")

    scheduler.step(val_metrics["pr_auc"])
    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    dt = time.time() - t0
    print(
        f"Epoch {epoch:02d}/{CFG.EPOCHS} ({dt:.1f}s) | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f}  val_PRAUC={val_metrics['pr_auc']:.4f}  "
        f"val_ROCAUC={val_metrics['roc_auc']:.4f}  val_Recall={val_metrics['recall']:.4f}  "
        f"val_F1={val_metrics['f1']:.4f}"
    )

    improved = ckpt_manager.update(model, optimizer, epoch, val_metrics, scheduler, scaler)
    epochs_without_improvement = 0 if improved else epochs_without_improvement + 1

    # fix: persist history.json after every epoch (not just once at the very end) so a
    # disconnect mid-run doesn't lose the metrics from already-completed epochs.
    with open(os.path.join(CFG.OUTPUT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    if epochs_without_improvement >= CFG.EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch} "
              f"(no PR-AUC improvement for {CFG.EARLY_STOP_PATIENCE} epochs).")
        break

print(f"\nBest Val PR-AUC:  {ckpt_manager.best_pr_auc:.4f}")
print(f"Best Val ROC-AUC: {ckpt_manager.best_roc_auc:.4f}")

No existing checkpoint found — starting fresh from epoch 1.


Epoch 1/50 [train]:   0%|          | 0/43 [00:00<?, ?it/s]

Epoch 1/50 [val]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01/50 (302.9s) | train_loss=0.0461 | val_loss=0.0436  val_PRAUC=0.7643  val_ROCAUC=0.7881  val_Recall=1.0000  val_F1=0.7708
  -> New best PR-AUC: 0.7643  (saved best_prauc.pt)
  -> New best ROC-AUC: 0.7881  (saved best_rocauc.pt)


Epoch 2/50 [train]:   0%|          | 0/43 [00:00<?, ?it/s]

...

Epoch 20/50 [val]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 20/50 (279.9s) | train_loss=0.0036 | val_loss=0.0777  val_PRAUC=0.8295  val_ROCAUC=0.8478  val_Recall=0.7838  val_F1=0.8056


Epoch 21/50 [train]:   0%|          | 0/43 [00:00<?, ?it/s]

Epoch 21/50 [val]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 21/50 (279.2s) | train_loss=0.0033 | val_loss=0.0814  val_PRAUC=0.8295  val_ROCAUC=0.8474  val_Recall=0.7838  val_F1=0.7945
Early stopping triggered at epoch 21 (no PR-AUC improvement for 8 epochs).

Best Val PR-AUC:  0.8460
Best Val ROC-AUC: 0.8634


### Load best checkpoint and tune clip threshold on validation set

Loads the best PR-AUC checkpoint, evaluates the validation set, and selects the clip-level decision threshold that maximizes validation F1.

In [16]:
import torch

# Load best PR-AUC checkpoint and run the VALIDATION set to find the best threshold
ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.SAVE_BEST_PRAUC)
print(f"Loading best PR-AUC weights from {ckpt_path} ...")
# fix: weights_only=False — same PyTorch 2.6 default-change issue as the test-eval cell
checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val PR-AUC={checkpoint['metrics']['pr_auc']:.4f})")

val_targets_all, val_probs_all = [], []
print("Evaluating Validation Set for threshold search...")
with torch.inference_mode():
    for clips, labels, paths in tqdm(val_loader, desc="Val Batches"):
        clips = clips.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(clips)
            probs = torch.sigmoid(logits)
        val_probs_all.extend(probs.cpu().float().numpy().tolist())
        val_targets_all.extend(labels.numpy().tolist())

best_thresh, best_f1 = best_threshold_for_f1(val_targets_all, val_probs_all)
print(f"Selected decision threshold (max F1 on val): {best_thresh:.3f}  (F1={best_f1:.4f})")

Loading best PR-AUC weights from /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/best_prauc.pt ...
Loaded checkpoint from epoch 13 (val PR-AUC=0.8460)
Evaluating Validation Set for threshold search...


Val Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Selected decision threshold (max F1 on val): 0.490  (F1=0.8116)


### Load best checkpoint and evaluate the test set

Reloads the best PR-AUC checkpoint and runs the held-out test set to collect probabilities, labels, and video paths for test metrics and later analysis.

In [17]:
import torch
from tqdm import tqdm

# 1. Create the lists your plotting/calibration cells are looking for
test_targets_all, test_probs_all, test_paths_all = [], [], []

# 2. Load your best PR-AUC checkpoint from the correct path
#    (CFG.OUTPUT_DIR + CFG.SAVE_BEST_PRAUC, not a hardcoded path)
ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.SAVE_BEST_PRAUC)
print(f"Loading best PR-AUC weights from {ckpt_path} ...")
# fix: PyTorch 2.6 changed torch.load's default to weights_only=True, which rejects
# the numpy scalars stored inside checkpoint["metrics"]. weights_only=False restores
# the old behavior — safe here since this is your own checkpoint from your own run.
checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val PR-AUC={checkpoint['metrics']['pr_auc']:.4f})")

# 3. Run the test set through the model
#    (each batch yields clips, labels, paths — 3 values, not 2 — since __getitem__
#     returns the source video path alongside the clip and label)
print("Evaluating Test Set...")
with torch.inference_mode():
    for clips, labels, paths in tqdm(test_loader, desc="Test Batches"):
        clips = clips.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(clips)
            probs = torch.sigmoid(logits)

        test_probs_all.extend(probs.cpu().float().numpy().tolist())
        test_targets_all.extend(labels.numpy().tolist())
        test_paths_all.extend(paths)

# 4. Compute the metrics your other cells (plotting, calibration) expect
test_metrics_default = compute_metrics(test_targets_all, test_probs_all, threshold=0.5)
test_metrics_tuned   = compute_metrics(test_targets_all, test_probs_all, threshold=best_thresh)

print("\n=== Test Set Metrics (threshold=0.50) ===")
for k, v in test_metrics_default.items():
    print(f"  {k}: {v:.4f}")
print(f"\n=== Test Set Metrics (tuned threshold={best_thresh:.3f}) ===")
for k, v in test_metrics_tuned.items():
    print(f"  {k}: {v:.4f}")

print("\n✓ Test evaluation complete. You can now run your plotting cell.")

Loading best PR-AUC weights from /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/best_prauc.pt ...
Loaded checkpoint from epoch 13 (val PR-AUC=0.8460)
Evaluating Test Set...


Test Batches: 100%|██████████| 5/5 [04:28<00:00, 53.60s/it]


=== Test Set Metrics (threshold=0.50) ===
  pr_auc: 0.9380
  roc_auc: 0.9225
  f1: 0.8800
  recall: 0.8684
  precision: 0.8919
  mse: 0.1135
  rmse: 0.3369

=== Test Set Metrics (tuned threshold=0.490) ===
  pr_auc: 0.9380
  roc_auc: 0.9225
  f1: 0.8684
  recall: 0.8684
  precision: 0.8684
  mse: 0.1135
  rmse: 0.3369

✓ Test evaluation complete. You can now run your plotting cell.


### Thresholded test metrics

Applies the selected threshold to test predictions and reports binary classification results.

In [18]:
y_true  = np.array(test_targets_all)
y_prob  = np.array(test_probs_all)
y_pred  = (y_prob >= best_thresh).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── (A) Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Suspicious"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion Matrix\n(threshold={best_thresh:.3f})", fontsize=13)

# ── (B) Precision-Recall Curve ───────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)
axes[1].plot(rec, prec, lw=2, color="steelblue", label=f"PR-AUC = {pr_auc:.3f}")
axes[1].scatter(
    [test_metrics_tuned["recall"]], [test_metrics_tuned["precision"]],
    marker="*", s=200, color="crimson", zorder=5,
    label=f"Operating point (F1={test_metrics_tuned['f1']:.3f})",
)
baseline = y_true.mean()
axes[1].axhline(baseline, ls="--", color="grey", label=f"Random baseline ({baseline:.2f})")
axes[1].set_xlabel("Recall");  axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve (Test Set)", fontsize=13)
axes[1].legend(fontsize=9);  axes[1].grid(alpha=0.3)

# ── (C) ROC Curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)
axes[2].plot(fpr, tpr, lw=2, color="darkorange", label=f"ROC-AUC = {roc_auc:.3f}")
axes[2].plot([0, 1], [0, 1], ls="--", color="grey", label="Random")
axes[2].set_xlabel("False Positive Rate");  axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("ROC Curve (Test Set)", fontsize=13)
axes[2].legend(fontsize=9);  axes[2].grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(CFG.OUTPUT_DIR, "evaluation_plots.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved evaluation plots → {fig_path}")

Saved evaluation plots → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/evaluation_plots.png


### Sliding-window video inference

Processes validation/test videos as ordered clip sequences and saves per-window suspicious probabilities for video-level and event-level evaluation.

In [19]:

def run_sliding_window_inference(records, model, cfg, device, max_windows_per_video=200, infer_batch_size=32):
    """
    Re-reads each video and runs the trained model over multiple sequential windows.

    This solves the clip-sampling/evaluation problem: instead of judging a whole video
    from one uniformly sampled clip, the notebook now scores many ordered windows and
    then aggregates them at video/event level.

    Runtime fix: this processes windows in mini-batches as they are read, instead of
    first storing all windows for a long video in memory. With 200 windows/video,
    storing all clip tensors at once can consume hundreds of MB on CPU RAM.
    """
    model.eval()
    window_span = cfg.NUM_FRAMES * max(cfg.INFER_STRIDE, 1)
    rows = []
    augmenter = ClipAugmenter(cfg.IMG_SIZE, cfg.MEAN, cfg.STD, train=False)

    with torch.inference_mode():
        for rec in records:
            path, label = rec["path"], rec["label"]

            cap = cv2.VideoCapture(path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()

            if total <= 0:
                print(f"[WARN] Could not read frame count for: {path}")
                continue

            n_windows = max(1, min(max_windows_per_video, math.ceil(total / window_span)))
            starts = [w * window_span for w in range(n_windows) if w * window_span < total]

            probs_all = []
            for start_i in range(0, len(starts), infer_batch_size):
                start_chunk = starts[start_i:start_i + infer_batch_size]

                clips = []
                for start in start_chunk:
                    frames, _ = read_frames_with_count(
                        path, cfg.NUM_FRAMES, cfg.IMG_SIZE,
                        mode="window", stride=cfg.INFER_STRIDE, start=start,
                    )
                    clips.append(augmenter(frames))

                if len(clips) == 0:
                    continue

                batch = torch.stack(clips, dim=0).to(device)
                with autocast(AMP_DEVICE_TYPE, enabled=AMP_ENABLED):
                    probs = torch.sigmoid(model(batch)).view(-1).cpu().float().numpy()

                probs_all.extend(probs.tolist())

                del clips, batch

            for w_idx, p in enumerate(probs_all):
                rows.append({
                    "path": path,
                    "label": label,
                    "class_name": rec.get("class_name", infer_class_name(path) if 'infer_class_name' in globals() else "Unknown"),
                    "window_idx": w_idx,
                    "prob": float(p),
                })

    return pd.DataFrame(rows)


def video_level_scores(window_df, top_k=3):
    """
    Converts window probabilities into one video-level score per video.

    score_max handles label noise: a suspicious video is positive if any window is strongly suspicious.
    score_topk_mean is less sensitive to one isolated false-positive spike.
    """
    rows = []
    for path, g in window_df.sort_values("window_idx").groupby("path", sort=False):
        probabilities = np.asarray(g["prob"].tolist(), dtype=np.float32)
        k = min(top_k, len(probabilities))
        topk_mean = float(np.mean(np.sort(probabilities)[-k:])) if k > 0 else 0.0
        rows.append({
            "path": path,
            "class_name": g["class_name"].iloc[0] if "class_name" in g else "Unknown",
            "true_label": int(g["label"].max()),
            "score_max": float(np.max(probabilities)) if len(probabilities) else 0.0,
            "score_topk_mean": topk_mean,
            "n_windows": int(len(probabilities)),
        })
    return pd.DataFrame(rows)


def compute_video_score_metrics(video_df, score_col):
    metrics = {
        "pr_auc": float(average_precision_score(video_df["true_label"], video_df[score_col])),
        "score_col": score_col,
        "n_videos": int(len(video_df)),
    }
    try:
        metrics["roc_auc"] = float(roc_auc_score(video_df["true_label"], video_df[score_col]))
    except ValueError:
        metrics["roc_auc"] = float("nan")
    return metrics


def run_and_save_window_eval(split_name, records):
    print(f"Running sliding-window inference over the {split_name} set "
          "(this re-reads each video; may take a few minutes) ...")
    window_df = run_sliding_window_inference(records, model, CFG, DEVICE)
    if window_df.empty:
        raise RuntimeError(
            f"No sliding-window predictions were generated for the {split_name} split. "
            "Check video paths, codecs, and max_windows_per_video."
        )

    print(f"Generated {len(window_df)} windowed clips across "
          f"{window_df['path'].nunique()} {split_name} videos "
          f"({window_df.groupby('path').size().mean():.1f} windows/video on average).")

    clip_path = os.path.join(CFG.OUTPUT_DIR, f"{split_name}_clip_probs.csv")
    window_df.to_csv(clip_path, index=False)
    print(f"Saved {split_name} per-clip probability sequence to {clip_path}")

    video_df = video_level_scores(window_df, top_k=3)
    score_path = os.path.join(CFG.OUTPUT_DIR, f"{split_name}_video_level_scores.csv")
    video_df.to_csv(score_path, index=False)

    max_metrics = compute_video_score_metrics(video_df, "score_max")
    topk_metrics = compute_video_score_metrics(video_df, "score_topk_mean")

    print(f"=== {split_name.upper()} Video-Level Scores ===")
    print(f"Max probability     PR-AUC={max_metrics['pr_auc']:.4f}  ROC-AUC={max_metrics['roc_auc']:.4f}")
    print(f"Top-3 mean          PR-AUC={topk_metrics['pr_auc']:.4f}  ROC-AUC={topk_metrics['roc_auc']:.4f}")
    print(f"Saved {split_name} video-level scores to {score_path}")
    return window_df, video_df, max_metrics, topk_metrics


# Run sliding-window validation first. This directly checks the label/clip-sampling problem
# on the validation split before touching the held-out test videos.
val_sliding_window_df, val_video_score_df, val_video_max_metrics, val_video_top3_metrics = run_and_save_window_eval(
    "val", val_records
)

# Run the same full-video evaluation on the held-out test split.
sliding_window_df, video_score_df, test_video_max_metrics, test_video_top3_metrics = run_and_save_window_eval(
    "test", test_records
)

# Backward-compatible filenames used by earlier analysis cells.
sliding_window_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "clip_probs.csv"), index=False)
video_score_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "video_level_max_scores.csv"), index=False)

# Backward-compatible variables used by final_summary.
video_prauc = test_video_max_metrics["pr_auc"]
video_rocauc = test_video_max_metrics["roc_auc"]


def simulate_deployment_alert(probs_in_order, ema_alpha=0.7, alert_thresh=0.80, n_consecutive=3):
    """
    Simulates deployment-time alerting logic over one video's clips, in temporal order.
    """
    if len(probs_in_order) == 0:
        return 0
    ema = probs_in_order[0]
    consecutive = 1 if ema >= alert_thresh else 0
    if consecutive >= n_consecutive:
        return 1
    for p in probs_in_order[1:]:
        ema = ema_alpha * p + (1 - ema_alpha) * ema
        if ema >= alert_thresh:
            consecutive += 1
            if consecutive >= n_consecutive:
                return 1
        else:
            consecutive = 0
    return 0


def event_level_metrics(window_df, ema_alpha=0.7, alert_thresh=0.80, n_consecutive=3, verbose=True):
    """
    Aggregates per-window predictions to per-video events using EMA + N consecutive clips.
    """
    grouped_records = []
    for path, g in window_df.sort_values("window_idx").groupby("path", sort=False):
        probs_in_order = g["prob"].tolist()
        alert_fired = simulate_deployment_alert(
            probs_in_order, ema_alpha=ema_alpha,
            alert_thresh=alert_thresh, n_consecutive=n_consecutive,
        )
        grouped_records.append({
            "path": path,
            "class_name": g["class_name"].iloc[0] if "class_name" in g else "Unknown",
            "true_label": int(g["label"].max()),
            "deployment_alert": int(alert_fired),
            "mean_prob": float(g["prob"].mean()),
            "max_prob": float(g["prob"].max()),
            "n_windows": int(len(g)),
        })

    grouped = pd.DataFrame(grouped_records)
    event_recall    = recall_score(grouped["true_label"], grouped["deployment_alert"], zero_division=0)
    event_precision = precision_score(grouped["true_label"], grouped["deployment_alert"], zero_division=0)
    event_f1        = f1_score(grouped["true_label"], grouped["deployment_alert"], zero_division=0)
    fp = ((grouped["true_label"] == 0) & (grouped["deployment_alert"] == 1)).sum()
    tn = ((grouped["true_label"] == 0) & (grouped["deployment_alert"] == 0)).sum()
    false_alarm_rate = fp / (fp + tn) if (fp + tn) else float("nan")

    metrics = {
        "event_recall": float(event_recall),
        "event_precision": float(event_precision),
        "event_f1": float(event_f1),
        "false_alarm_rate": float(false_alarm_rate),
        "ema_alpha": float(ema_alpha),
        "alert_thresh": float(alert_thresh),
        "n_consecutive": int(n_consecutive),
    }

    if verbose:
        print(f"=== Event-Level Metrics (EMA={ema_alpha}, thresh={alert_thresh}, N={n_consecutive}) ===")
        print(f"  Videos evaluated : {len(grouped)}")
        print(f"  Suspicious videos: {grouped['true_label'].sum()} ({100*grouped['true_label'].mean():.1f}%)")
        print(f"  Avg windows/video: {grouped['n_windows'].mean():.1f}")
        print(f"  Event Recall     : {event_recall:.4f}")
        print(f"  Event Precision  : {event_precision:.4f}")
        print(f"  Event F1         : {event_f1:.4f}")
        print(f"  False Alarm Rate : {false_alarm_rate:.4f}")

    return grouped, metrics


# Untuned baseline/fixed setting; the validation sweep in the next cell will choose final alert params.
event_df, event_metrics = event_level_metrics(
    sliding_window_df, ema_alpha=0.5, alert_thresh=0.65, n_consecutive=2,
)
event_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "event_predictions_fixed_test.csv"), index=False)

print("\n--- Old alert-logic settings (alpha=0.7, thresh=0.80, N=3) on coverage-fixed test data ---")
_, _ = event_level_metrics(
    sliding_window_df, ema_alpha=0.7, alert_thresh=0.8, n_consecutive=3,
)


Running sliding-window inference over the val set (this re-reads each video; may take a few minutes) ...
Generated 8561 windowed clips across 75 val videos (114.1 windows/video on average).
Saved val per-clip probability sequence to /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/val_clip_probs.csv
=== VAL Video-Level Scores ===
Max probability     PR-AUC=0.8604  ROC-AUC=0.8439
Top-3 mean          PR-AUC=0.8667  ROC-AUC=0.8542
Saved val video-level scores to /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/val_video_level_scores.csv
Running sliding-window inference over the test set (this re-reads each video; may take a few minutes) ...
Generated 7425 windowed clips across 75 test videos (99.0 windows/video on average).
Saved test per-clip probability sequence to /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/test_clip_probs.csv
=== TEST Video-Level Scores ===
Max probability     PR-AUC=0.8963  ROC-AUC=0.9018
Top-3 mean          PR-AUC

## Alert-Logic Parameter Sweep

After fixing the `max_windows_per_video` coverage bug above, this cell tests whether a
different `(alpha, threshold, N)` combination for the deployment alert logic improves
event-level recall -- especially for **Fighting**, the weakest class -- without retraining
the model. This runs entirely on `sliding_window_df` already in memory from the cell above,
so results reflect the corrected (full-coverage) data, not the previously truncated run.


**Alert-logic tuning note — validation first, test once:**

The previous version tuned `(alpha, threshold, N)` directly on the test sliding-window output.
That could make event-level test metrics too optimistic.

This version first runs sliding-window inference on the **validation** videos, selects the alert
setting on validation results, and then applies the selected setting once to the held-out
**test** videos. This is the cleaner reporting approach.


### Validation-tuned alert sweep

Tunes EMA threshold and consecutive-window alert settings on validation videos, then applies the selected setting to test videos once.

In [20]:

# ── Alert-Logic Parameter Sweep (VALIDATION-tuned, TEST-applied) ─────────────
# The old version swept alert parameters on the test set, which makes test event metrics
# optimistic. This version selects alpha/threshold/N on val_sliding_window_df and applies
# the selected setting once to the held-out test sliding_window_df.

import numpy as np
import pandas as pd
from itertools import product
from collections import Counter


def infer_class_name(path):
    if "/Burglary/" in path:
        return "Burglary"
    if "/Fighting/" in path:
        return "Fighting"
    if "/Stealing/" in path:
        return "Stealing"
    if "Normal" in path:
        return "Normal"
    return "Unknown"


def build_clip_prob_dict(window_df):
    clip_probs = {}
    labels = {}
    for path, g in window_df.sort_values("window_idx").groupby("path", sort=False):
        clip_probs[path] = g["prob"].tolist()
        labels[path] = {
            "true_label": int(g["label"].iloc[0]),
            "class_name": g["class_name"].iloc[0] if "class_name" in g else infer_class_name(path),
        }
    return clip_probs, labels


def simulate_alert(probs, alpha, threshold, N, cooldown_clips=0):
    ema = None
    consecutive = 0
    cooldown_remaining = 0
    fired = False
    first_fire_idx = None
    n_alerts = 0
    for i, p in enumerate(probs):
        ema = p if ema is None else alpha * p + (1 - alpha) * ema
        if cooldown_remaining > 0:
            cooldown_remaining -= 1
            consecutive = consecutive + 1 if ema >= threshold else 0
            continue
        if ema >= threshold:
            consecutive += 1
        else:
            consecutive = 0
        if consecutive >= N:
            fired = True
            n_alerts += 1
            if first_fire_idx is None:
                first_fire_idx = i
            cooldown_remaining = cooldown_clips
            consecutive = 0
    return fired, first_fire_idx, n_alerts


def evaluate_settings(clip_probs, labels, alpha, threshold, N, cooldown_clips=0):
    rows = []
    for video_path, probs in clip_probs.items():
        meta = labels[video_path]
        fired, first_fire_idx, n_alerts = simulate_alert(probs, alpha, threshold, N, cooldown_clips)
        rows.append({
            "video": video_path,
            "class_name": meta["class_name"],
            "true_label": meta["true_label"],
            "alert_fired": int(fired),
            "n_alerts": n_alerts,
            "first_fire_idx": first_fire_idx,
            "n_clips": len(probs),
        })
    df = pd.DataFrame(rows)
    tp = ((df.true_label == 1) & (df.alert_fired == 1)).sum()
    fn = ((df.true_label == 1) & (df.alert_fired == 0)).sum()
    fp = ((df.true_label == 0) & (df.alert_fired == 1)).sum()
    tn = ((df.true_label == 0) & (df.alert_fired == 0)).sum()
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    f1 = (2 * precision * recall / (precision + recall)
          if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0 else np.nan)
    far = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    overall = {
        "alpha": alpha,
        "threshold": threshold,
        "N": N,
        "cooldown_clips": cooldown_clips,
        "event_recall": recall,
        "event_precision": precision,
        "event_f1": f1,
        "false_alarm_rate": far,
        "tp": int(tp), "fn": int(fn), "fp": int(fp), "tn": int(tn),
    }

    per_class = {}
    for cls in sorted(df.class_name.unique()):
        sub = df[df.class_name == cls]
        if sub.true_label.iloc[0] == 1:
            per_class[cls] = {"n": len(sub), "recall": float(sub.alert_fired.mean())}
        else:
            per_class[cls] = {"n": len(sub), "false_alarm_rate": float(sub.alert_fired.mean())}
    return overall, per_class, df


val_clip_probs, val_labels = build_clip_prob_dict(val_sliding_window_df)
test_clip_probs, test_labels = build_clip_prob_dict(sliding_window_df)

print(f"Prepared {len(val_clip_probs)} validation videos and {len(test_clip_probs)} test videos for alert tuning.")
print("Val class counts:", dict(Counter(v["class_name"] for v in val_labels.values())))
print("Test class counts:", dict(Counter(v["class_name"] for v in test_labels.values())))

ALPHAS = [0.3, 0.5, 0.7, 0.9]
THRESHOLDS = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
N_VALUES = [1, 2, 3]
COOLDOWN_CLIPS = 0

sweep_rows = []
per_class_rows = []
for alpha, threshold, N in product(ALPHAS, THRESHOLDS, N_VALUES):
    overall, per_class, _df = evaluate_settings(
        val_clip_probs, val_labels, alpha, threshold, N, COOLDOWN_CLIPS
    )
    row = dict(overall)
    for cls, vals in per_class.items():
        for metric_name, metric_value in vals.items():
            row[f"{metric_name}_{cls}"] = metric_value
    sweep_rows.append(row)

results_df = pd.DataFrame(sweep_rows)
results_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "alert_sweep_validation.csv"), index=False)

# Selection rule: maximize validation event F1, then recall, then minimize false alarms.
ranked = results_df.sort_values(
    by=["event_f1", "event_recall", "false_alarm_rate"],
    ascending=[False, False, True],
).reset_index(drop=True)

best_alert = ranked.iloc[0].to_dict()
best_alert_params = {
    "alpha": float(best_alert["alpha"]),
    "threshold": float(best_alert["threshold"]),
    "N": int(best_alert["N"]),
    "cooldown_clips": int(best_alert["cooldown_clips"]),
    "selection_split": "validation",
    "selection_rule": "max val event_f1, then max recall, then min false_alarm_rate",
}

print("\nTop validation alert settings:")
cols_to_show = ["alpha", "threshold", "N", "event_recall", "event_precision", "event_f1", "false_alarm_rate", "tp", "fn", "fp", "tn"]
print(ranked[cols_to_show].head(10).to_string(index=False))
print("\nSelected alert params from validation:", best_alert_params)

# Apply the selected validation-tuned parameters to the held-out test set exactly once.
val_overall, val_per_class, val_event_df = evaluate_settings(
    val_clip_probs, val_labels,
    best_alert_params["alpha"], best_alert_params["threshold"], best_alert_params["N"], best_alert_params["cooldown_clips"]
)
test_overall, test_per_class, event_df = evaluate_settings(
    test_clip_probs, test_labels,
    best_alert_params["alpha"], best_alert_params["threshold"], best_alert_params["N"], best_alert_params["cooldown_clips"]
)

val_event_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "event_predictions_val_validation_tuned.csv"), index=False)
event_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "event_predictions.csv"), index=False)

# Keep final_summary variable names backward-compatible, but now they are test metrics
# from a validation-selected alert setting, not test-swept settings.
event_metrics = {
    "event_recall": float(test_overall["event_recall"]),
    "event_precision": float(test_overall["event_precision"]),
    "event_f1": float(test_overall["event_f1"]),
    "false_alarm_rate": float(test_overall["false_alarm_rate"]),
    "tp": int(test_overall["tp"]), "fn": int(test_overall["fn"]),
    "fp": int(test_overall["fp"]), "tn": int(test_overall["tn"]),
    "alpha": best_alert_params["alpha"],
    "threshold": best_alert_params["threshold"],
    "N": best_alert_params["N"],
}

print("\n=== Validation result at selected alert setting ===")
print(val_overall)
print("Per-class:", val_per_class)

print("\n=== Held-out TEST result using validation-selected alert setting ===")
print(test_overall)
print("Per-class:", test_per_class)

# Plot validation trade-off for Fighting recall vs Normal false alarms.
fig, ax = plt.subplots(figsize=(8, 6))
for N_val in N_VALUES:
    sub = results_df[results_df.N == N_val]
    y_col = "recall_Fighting" if "recall_Fighting" in sub.columns else "event_recall"
    ax.scatter(sub["false_alarm_rate"], sub[y_col], label=f"N={N_val}", alpha=0.6)
ax.scatter([best_alert["false_alarm_rate"]], [best_alert.get("recall_Fighting", best_alert["event_recall"])],
           marker="*", s=300, label="Selected on val", zorder=5)
ax.set_xlabel("False Alarm Rate on Normal videos (validation)")
ax.set_ylabel("Fighting Event Recall (validation)")
ax.set_title("Validation Alert Sweep: Fighting Recall vs False Alarm Rate")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "validation_fighting_recall_tradeoff.png"), dpi=150)
plt.show()


Prepared 75 validation videos and 75 test videos for alert tuning.
Val class counts: {'Normal': 38, 'Stealing': 15, 'Burglary': 15, 'Fighting': 7}
Test class counts: {'Normal': 37, 'Stealing': 15, 'Burglary': 15, 'Fighting': 8}

Top validation alert settings:
 alpha  threshold  N  event_recall  event_precision  event_f1  false_alarm_rate  tp  fn  fp  tn
   0.3       0.60  3      0.810811         0.882353  0.845070          0.105263  30   7   4  34
   0.5       0.60  3      0.810811         0.882353  0.845070          0.105263  30   7   4  34
   0.5       0.55  3      0.837838         0.837838  0.837838          0.157895  31   6   6  32
   0.7       0.55  3      0.837838         0.837838  0.837838          0.157895  31   6   6  32
   0.7       0.60  2      0.837838         0.837838  0.837838          0.157895  31   6   6  32
   0.7       0.60  3      0.810811         0.857143  0.833333          0.131579  30   7   5  33
   0.9       0.60  3      0.810811         0.857143  0.833333       

### Calibration analysis

Computes calibration error and plots how reliable the model probabilities are as confidence scores.

In [21]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Standard ECE: bins predictions by confidence, compares each bin's mean predicted
    probability against its empirical accuracy (fraction of positives), weights by bin size.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    if len(y_prob) == 0:
        raise ValueError("Cannot compute calibration error on an empty prediction array.")
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)
        n_in_bin = mask.sum()
        if n_in_bin == 0:
            bin_stats.append({"bin_lo": lo, "bin_hi": hi, "n": 0,
                              "mean_pred": np.nan, "empirical_acc": np.nan})
            continue
        mean_pred  = y_prob[mask].mean()
        emp_acc    = y_true[mask].mean()
        ece       += (n_in_bin / len(y_prob)) * abs(mean_pred - emp_acc)
        bin_stats.append({"bin_lo": lo, "bin_hi": hi, "n": int(n_in_bin),
                          "mean_pred": float(mean_pred), "empirical_acc": float(emp_acc)})
    return float(ece), bin_stats


ece, calib_bins = expected_calibration_error(test_targets_all, test_probs_all, n_bins=10)
calib_df = pd.DataFrame(calib_bins)
print(f"Expected Calibration Error (ECE) on test set: {ece:.4f}")
print("(Lower is better; <0.05 is generally considered well-calibrated.)\n")
print(calib_df.to_string(index=False))

# ── Reliability diagram ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
valid = calib_df.dropna(subset=["mean_pred", "empirical_acc"])
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="Perfect calibration")
ax.plot(valid["mean_pred"], valid["empirical_acc"], marker="o", color="steelblue",
        label=f"Model (ECE={ece:.3f})")
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Empirical fraction positive (per bin)")
ax.set_title("Reliability Diagram (Test Set)")
ax.axvline(best_thresh, ls=":", color="crimson", alpha=0.7, label=f"Selected clip threshold ({best_thresh:.2f})")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
calib_fig_path = os.path.join(CFG.OUTPUT_DIR, "calibration_diagram.png")
plt.savefig(calib_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved reliability diagram → {calib_fig_path}")

with open(os.path.join(CFG.OUTPUT_DIR, "calibration_metrics.json"), "w") as f:
    json.dump({"ece": ece, "bins": calib_bins}, f, indent=2)


Expected Calibration Error (ECE) on test set: 0.1406
(Lower is better; <0.05 is generally considered well-calibrated.)

 bin_lo  bin_hi  n  mean_pred  empirical_acc
    0.0     0.1  7   0.070666       0.142857
    0.1     0.2 11   0.161349       0.090909
    0.2     0.3 12   0.245514       0.000000
    0.3     0.4  2   0.339478       0.000000
    0.4     0.5  6   0.452189       0.500000
    0.5     0.6  6   0.545898       1.000000
    0.6     0.7  9   0.641927       0.666667
    0.7     0.8  7   0.755929       0.857143
    0.8     0.9 10   0.852490       1.000000
    0.9     1.0  5   0.956055       1.000000
Saved reliability diagram → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/calibration_diagram.png


### PyTorch CPU benchmark

Measures model latency and FPS on CPU before OpenVINO conversion.

In [22]:
def benchmark_model_cpu(every_n_blocks, num_frames=CFG.NUM_FRAMES,
                        img_size=CFG.IMG_SIZE, n_warmup=5, n_runs=20):
    """
    Benchmark latency of a freshly initialised clip model on CPU.
    every_n_blocks=None means no TSM, but the model still processes all T frames and
    temporal-averages them. This makes the No-TSM latency comparable with TSM settings.
    """
    m = SuspiciousActivityModel(
        num_frames=num_frames, dropout=0.0, pretrained=False,
        every_n_blocks=every_n_blocks,
    ).cpu().eval()

    dummy = torch.zeros(1, num_frames, 3, img_size, img_size)  # batch=1, full video clip

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = m(dummy)

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0  = time.perf_counter()
            _ = m(dummy)
            latencies.append((time.perf_counter() - t0) * 1000)   # ms

    avg_ms = float(np.mean(latencies))
    std_ms = float(np.std(latencies))
    fps    = 1000.0 / avg_ms
    return avg_ms, std_ms, fps


ablation_settings = [
    ("No TSM",       None),
    ("Every block",  1),
    ("Every 2 blocks", 2),
    ("Every 3 blocks", 3),
]

print("Running TSM frequency ablation (CPU latency benchmark) ...")
print(f"{'Setting':<20}  {'Avg latency':>13}  {'Std':>8}  {'FPS':>8}")
print("-" * 55)

ablation_results = []
for label, every_n in ablation_settings:
    avg_ms, std_ms, fps = benchmark_model_cpu(every_n)

    ablation_results.append({"setting": label, "avg_latency_ms": avg_ms,
                              "std_ms": std_ms, "fps": fps})
    print(f"  {label:<18}  {avg_ms:>10.1f} ms  {std_ms:>6.1f} ms  {fps:>6.1f}")

ablation_df = pd.DataFrame(ablation_results)
print("\nNote: PR-AUC differences must be measured by retraining with each setting and comparing")
print("val PR-AUC scores.  The latency figures above are sufficient for the CPU tradeoff analysis.")
print("Recommended: use every_n_blocks=2 if FPS difference exceeds 1.5× and PR-AUC drop < 0.01.")


Running TSM frequency ablation (CPU latency benchmark) ...
Setting                 Avg latency       Std       FPS
-------------------------------------------------------
  No TSM                    53.1 ms     1.0 ms    18.8
  Every block               58.9 ms     1.2 ms    17.0
  Every 2 blocks            58.0 ms     3.0 ms    17.2
  Every 3 blocks            56.4 ms     1.3 ms    17.7

Note: PR-AUC differences must be measured by retraining with each setting and comparing
val PR-AUC scores.  The latency figures above are sufficient for the CPU tradeoff analysis.
Recommended: use every_n_blocks=2 if FPS difference exceeds 1.5× and PR-AUC drop < 0.01.


### Trained PyTorch CPU benchmark with optional RAM tracking

Benchmarks the trained PyTorch model on CPU and reports latency, FPS, and RAM usage. If `psutil` is unavailable, RAM tracking is skipped without stopping the notebook.

In [23]:
# fix: psutil is preinstalled on standard Kaggle images, but importing it bare (no
# fallback) means this cell hard-crashes on any environment where it's missing,
# unlike the onnx/openvino imports later in the notebook which already degrade
# gracefully. RAM measurement becomes a no-op if psutil isn't available, but the
# latency/FPS benchmark (the part that actually gates the deployment target) still runs.
try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    print("[INFO] psutil not installed; RAM delta will be reported as None. "
          "Run pip_install('psutil') above and re-run this cell to enable it.")
    PSUTIL_AVAILABLE = False

def benchmark_trained_model_cpu(model, num_frames=CFG.NUM_FRAMES, img_size=CFG.IMG_SIZE,
                                 n_warmup=5, n_runs=30):
    """
    Benchmarks the *trained* model on CPU (force-moves to cpu for the test).
    Measures latency, FPS, and RAM footprint.
    """
    model_cpu = model.cpu().eval()
    dummy     = torch.zeros(1, num_frames, 3, img_size, img_size)

    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            model_cpu(dummy)

    # Measure RAM before
    proc = psutil.Process(os.getpid()) if PSUTIL_AVAILABLE else None
    ram_before_mb = proc.memory_info().rss / 1024**2 if proc else None

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model_cpu(dummy)
            latencies.append((time.perf_counter() - t0) * 1000)

    ram_after_mb = proc.memory_info().rss / 1024**2 if proc else None

    avg_ms  = float(np.mean(latencies))
    p95_ms  = float(np.percentile(latencies, 95))
    fps     = 1000.0 / avg_ms
    ram_mb  = (ram_after_mb - ram_before_mb) if (ram_after_mb is not None) else None

    # Move model back to original device
    model.to(DEVICE)

    return {
        "avg_latency_ms": avg_ms,
        "p95_latency_ms": p95_ms,
        "fps":            fps,
        "ram_delta_mb":   ram_mb,
    }


bench = benchmark_trained_model_cpu(model)

print("=" * 50)
print("  Deployment Benchmark — Trained Model on CPU")
print("=" * 50)
print(f"  Average latency : {bench['avg_latency_ms']:.1f} ms/clip  "
      f"({'✓ PASS' if bench['avg_latency_ms'] < 200 else '✗ FAIL — exceeds 200 ms target'})")
print(f"  P95 latency     : {bench['p95_latency_ms']:.1f} ms/clip")
print(f"  Effective FPS   : {bench['fps']:.1f}  "
      f"({'✓ PASS' if bench['fps'] >= 5 else '✗ FAIL — below 5 FPS target'})")
if bench["ram_delta_mb"] is not None:
    print(f"  RAM delta       : {bench['ram_delta_mb']:.1f} MB  "
          f"({'✓ PASS' if bench['ram_delta_mb'] < 4096 else '✗ FAIL — exceeds 4 GB target'})")
else:
    print("  RAM delta       : N/A (psutil not installed)")
print("=" * 50)

bench_path = os.path.join(CFG.OUTPUT_DIR, "deployment_benchmark.json")
with open(bench_path, "w") as f:
    json.dump(bench, f, indent=2)
print(f"Benchmark results saved → {bench_path}")

  Deployment Benchmark — Trained Model on CPU
  Average latency : 60.6 ms/clip  (✓ PASS)
  P95 latency     : 62.9 ms/clip
  Effective FPS   : 16.5  (✓ PASS)
  RAM delta       : 0.0 MB  (✓ PASS)
Benchmark results saved → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/deployment_benchmark.json


### ONNX export

Exports the trained PyTorch model to ONNX so it can be converted for CPU deployment.

In [24]:
# ── Step 1: Export to ONNX ───────────────────────────────────────────────────
onnx_path = os.path.join(CFG.OUTPUT_DIR, "model.onnx")

model.eval().cpu()

dummy_input = torch.zeros(1, CFG.NUM_FRAMES, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)

# fix: torch.onnx.export now defaults to the "dynamo" exporter, which requires the
# onnxscript package — not installed in this environment, causing
# ModuleNotFoundError: No module named 'onnxscript'. dynamo=False forces the older
# TorchScript-based exporter, which has no such dependency and is what dynamic_axes
# (a legacy-exporter argument) was written for.
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    dynamo           = False,
    opset_version    = 12,
    input_names      = ["clip"],
    output_names     = ["logit"],
    dynamic_axes     = {"clip": {0: "batch_size"}},   # allow variable batch at inference
    do_constant_folding = True,
    verbose          = False,
)
print(f"ONNX model saved → {onnx_path}")

# Quick sanity check: verify the ONNX graph is valid
try:
    import onnx
except ImportError:
    print("[INFO] onnx package not installed. Installing it now for ONNX validation...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx"], check=True)
    import onnx

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✓ ONNX graph check passed.")

model.to(DEVICE)   # restore to training device

/tmp/ipykernel_24/1715357851.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_24/1955722361.py:94: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert t == self.num_frames, f"Expected {self.num_frames} frames, got {t}"
/tmp/ipykernel_24/1955722361.py:14: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this

ONNX model saved → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/model.onnx
✓ ONNX graph check passed.


SuspiciousActivityModel(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): TSMBlockWrapper(
      (shift): TemporalShift()
      (block): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          )
        )
      )
    )
    (2): TSMBlockWrapper(
      (shift): TemporalSh

### ONNX export file check

Checks that the ONNX file was created correctly, prints its file size, and lists the current output directory contents before validation/conversion.

In [25]:
import os
print("onnx_path:", onnx_path)
print("Exists:", os.path.exists(onnx_path))
print("Is file:", os.path.isfile(onnx_path) if os.path.exists(onnx_path) else "N/A")
if os.path.exists(onnx_path):
    print("Size (bytes):", os.path.getsize(onnx_path))
print()
print("Contents of CFG.OUTPUT_DIR:")
print(os.listdir(CFG.OUTPUT_DIR))

onnx_path: /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/model.onnx
Exists: True
Is file: True
Size (bytes): 12618806

Contents of CFG.OUTPUT_DIR:
['event_predictions_val_validation_tuned.csv', 'latest.pt', 'best_prauc.pt', 'deployment_benchmark.json', 'model.onnx', 'calibration_diagram.png', 'event_predictions_fixed_test.csv', 'validation_fighting_recall_tradeoff.png', 'history.json', 'alert_sweep_validation.csv', 'val_video_level_scores.csv', 'event_predictions.csv', 'video_level_max_scores.csv', 'test_video_level_scores.csv', 'evaluation_plots.png', 'val_clip_probs.csv', 'clip_probs.csv', 'best_rocauc.pt', 'test_clip_probs.csv', 'calibration_metrics.json']


### OpenVINO import/version check

Imports OpenVINO for deployment conversion. If it is missing, the cell installs it and then prints the installed version.

In [26]:
try:
    import openvino as ov
except ImportError:
    print("[INFO] openvino is not installed. Installing it now for export/deployment...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openvino"], check=True)
    import openvino as ov
print(ov.__version__)

[INFO] openvino is not installed. Installing it now for export/deployment...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 MB 32.7 MB/s eta 0:00:00
2026.2.1-21919-ede283a88e3-releases/2026/2


### ONNX validation

Checks that the exported ONNX model is valid before converting it to OpenVINO.

In [27]:
try:
    import onnx
except ImportError:
    print("[INFO] onnx is not installed. Installing it now for ONNX validation...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx"], check=True)
    import onnx

try:
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("✓ File loads as valid ONNX and passes checker.")
    print("IR version:", onnx_model.ir_version)
    print("Opset:", [f"{o.domain or 'ai.onnx'}:{o.version}" for o in onnx_model.opset_import])
except Exception as e:
    print("✗ Failed to load/validate as ONNX:", repr(e))


✓ File loads as valid ONNX and passes checker.
IR version: 7
Opset: ['ai.onnx:12']


### OpenVINO frontend availability check

Checks the available OpenVINO frontends so the notebook can confirm ONNX-to-OpenVINO conversion support before conversion.

In [28]:
try:
    import openvino as ov
except ImportError:
    print("[INFO] openvino is not installed. Installing it now for export/deployment...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openvino"], check=True)
    import openvino as ov
print(ov.__version__)
from openvino.frontend import FrontEndManager
fem = FrontEndManager()
print(fem.get_available_front_ends())

2026.2.1-21919-ede283a88e3-releases/2026/2
['tflite', 'onnx', 'paddle', 'pytorch', 'ir', 'jax', 'tf']


### OpenVINO FP32 conversion and benchmark

Converts the ONNX model to OpenVINO IR and benchmarks FP32 CPU inference speed.

In [29]:
# ── Step 2: Convert ONNX → OpenVINO IR ───────────────────────────────────────
import os
import time
import json
import numpy as np
import torch
try:
    import openvino as ov
except ImportError:
    print("[INFO] openvino is not installed. Installing it now for export/deployment...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openvino"], check=True)
    import openvino as ov

ir_dir = os.path.join(CFG.OUTPUT_DIR, "openvino_ir")
os.makedirs(ir_dir, exist_ok=True)
ir_xml = os.path.join(ir_dir, "model.xml")

# Defensive: rebuild these if this cell is run standalone or after a kernel restart,
# rather than assuming they survived from the ONNX export cell.
onnx_path = os.path.join(CFG.OUTPUT_DIR, "model.onnx")
dummy_input = torch.zeros(1, CFG.NUM_FRAMES, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)

print("Starting OpenVINO conversion (Modern API)...")
ov_model = ov.convert_model(onnx_path)
ov.save_model(ov_model, ir_xml, compress_to_fp16=False)
print(f"✓ OpenVINO FP32 IR saved → {ir_xml}")

print("Starting OpenVINO CPU Benchmark...")
core = ov.Core()
net = core.compile_model(ov_model, device_name="CPU")
infer_request = net.create_infer_request()
dummy_np = dummy_input.numpy()

for _ in range(5):
    infer_request.infer({"clip": dummy_np})

ov_lats = []
for _ in range(30):
    t0 = time.perf_counter()
    infer_request.infer({"clip": dummy_np})
    ov_lats.append((time.perf_counter() - t0) * 1000)

ov_avg_ms = float(np.mean(ov_lats))
ov_fps    = 1000.0 / ov_avg_ms
speedup   = bench["avg_latency_ms"] / ov_avg_ms if 'bench' in locals() else 0.0

print(f"  OpenVINO FP32 avg latency : {ov_avg_ms:.1f} ms  ({ov_fps:.1f} FPS)")
if speedup > 0:
    print(f"  Speedup vs PyTorch CPU    : {speedup:.2f}×")

ov_bench = {"ov_fp32_avg_ms": ov_avg_ms, "ov_fp32_fps": ov_fps, "speedup_vs_pytorch": speedup}
with open(os.path.join(CFG.OUTPUT_DIR, "openvino_benchmark.json"), "w") as f:
    json.dump(ov_bench, f, indent=2)

Starting OpenVINO conversion (Modern API)...
✓ OpenVINO FP32 IR saved → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/openvino_ir/model.xml
Starting OpenVINO CPU Benchmark...
✓ OpenVINO FP32 IR saved → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/openvino_ir/model.xml
Starting OpenVINO CPU Benchmark...
  OpenVINO FP32 avg latency : 38.0 ms  (26.3 FPS)
  Speedup vs PyTorch CPU    : 1.59×


### Torch runtime check before OpenVINO benchmarking

Prints the active PyTorch version and CUDA status, then runs a tiny tensor check to confirm the runtime is still healthy before deployment benchmarking.

In [30]:
import torch
print(torch.__version__, torch.cuda.is_available())
x = torch.randn(2, 2).cuda() if torch.cuda.is_available() else torch.randn(2, 2)
print(x)

2.10.0+cu128 True
tensor([[-1.5083,  0.3320],
        [-0.8907, -0.3873]], device='cuda:0')


### INT8 quantization

Creates an INT8 OpenVINO model using calibration data to reduce model size and improve CPU deployment efficiency.

In [31]:
# ── Step 3: INT8 Quantization (Post-Training Quantization via NNCF) ─────────

try:
    import nncf
except ImportError:
    print("[INFO] nncf is not installed. Installing it now for INT8 quantization...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nncf"], check=True)
    import nncf

try:
    import openvino as ov
except ImportError:
    print("[INFO] openvino is not installed. Installing it now for INT8 quantization...")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openvino"], check=True)
    import openvino as ov

# NNCF needs a transform_fn that takes one sample from your dataloader and returns
# exactly what the model's forward() expects as input — here, just the clips tensor
# (drop the label and path, which val_loader also yields per the dataset's __getitem__).
def transform_fn(data_item):
    clips, labels, paths = data_item
    return clips.numpy()

# Use your existing validation set as the calibration dataset — it's already a
# representative sample of real data, and matches the design doc's plan to validate
# INT8 against the same validation set used during training.
calibration_dataset = nncf.Dataset(val_loader, transform_fn)

print("Running NNCF INT8 post-training quantization (this may take a few minutes)...")
ov_model_fp32 = ov.Core().read_model(ir_xml)  # re-load the FP32 IR we just saved
ov_model_int8 = nncf.quantize(ov_model_fp32, calibration_dataset)

ir_xml_int8 = os.path.join(ir_dir, "model_int8.xml")
ov.save_model(ov_model_int8, ir_xml_int8)
print(f"✓ OpenVINO INT8 IR saved → {ir_xml_int8}")

[INFO] nncf is not installed. Installing it now for INT8 quantization...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
torch 2.10.0+cu128 requires cuda-bindings==12.9.4; platform_system == "Linux", but you have cuda-bindings 13.2.0 which is incompatible.


Running NNCF INT8 post-training quantization (this may take a few minutes)...


Output()

Output()

✓ OpenVINO INT8 IR saved → /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/openvino_ir/model_int8.xml


### INT8 accuracy validation

Compares INT8 validation metrics with FP32 metrics to decide whether quantization is acceptable.

In [32]:
# ── Step 4: Validate INT8 accuracy against FP32 (design doc requires ≤2% PR-AUC drop) ─
def compute_int8_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    pr_auc = average_precision_score(y_true, y_prob)
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = float("nan")

    return {
        "pr_auc":    pr_auc,
        "roc_auc":   roc_auc,
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
    }
core_int8 = ov.Core()
compiled_int8 = core_int8.compile_model(ov_model_int8, device_name="CPU")
infer_int8 = compiled_int8.create_infer_request()

int8_targets, int8_probs = [], []
with torch.inference_mode():
    for clips, labels, paths in tqdm(val_loader, desc="INT8 Val Batches"):
        out = infer_int8.infer({"clip": clips.numpy()})
        logits = list(out.values())[0]
        probs = 1 / (1 + np.exp(-logits))  # manual sigmoid, since this is raw numpy now
        int8_probs.extend(probs.flatten().tolist())
        int8_targets.extend(labels.numpy().tolist())

int8_metrics = compute_int8_metrics(int8_targets, int8_probs, threshold=0.5)
fp32_val_prauc = ckpt_manager.best_pr_auc  # your FP32 model's validation PR-AUC

if fp32_val_prauc is None or fp32_val_prauc <= 0:
    print("[WARN] FP32 validation PR-AUC is not available/valid; INT8 drop percentage cannot be computed reliably.")
    drop = float("nan")
    drop_pct = float("nan")
else:
    drop = fp32_val_prauc - int8_metrics["pr_auc"]
    drop_pct = (drop / fp32_val_prauc) * 100

print(f"FP32 val PR-AUC : {fp32_val_prauc:.4f}")
print(f"INT8 val PR-AUC : {int8_metrics['pr_auc']:.4f}")
print(f"Drop            : {drop_pct:.2f}%  "
      f"({'✓ PASS — within 2% tolerance' if drop_pct <= 2.0 else '✗ FAIL — exceeds 2% tolerance'})")

INT8 Val Batches: 100%|██████████| 5/5 [02:07<00:00, 25.45s/it]

FP32 val PR-AUC : 0.8460
INT8 val PR-AUC : 0.8258
Drop            : 2.39%  (✗ FAIL — exceeds 2% tolerance)


### Evaluation loader helper

Builds validation/test loaders safely for normal evaluation, ablation, and cross-validation sections.

In [33]:
def make_eval_loader(dataset, cfg, generator=None):
    """Evaluation loader: keeps the final partial batch; no samples are discarded.
    fix: generator is now an explicit parameter (defaulting to the global `g`) instead of
    being hardcoded to the outer-scope global. Without this, every eval loader silently
    shared the global generator regardless of which fold created it — inconsistent with
    the fold-specific seeding (g_fold) used for that fold's train loader."""
    return DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator if generator is not None else g,
        persistent_workers=(cfg.NUM_WORKERS > 0),
    )


def run_cv_fold(fold_idx, train_recs, val_recs, cfg, device, motion_cache, n_epochs=20):
    """
    Trains a fresh model for one CV fold.  Uses a reduced epoch budget (20) for speed;
    increase to cfg.EPOCHS for final CV results.
    """
    fold_train_ds = UCFCrimeClipDataset(train_recs, cfg, train=True)
    fold_val_ds   = UCFCrimeClipDataset(val_recs,   cfg, train=False)

    fold_sampler, _ = build_class_balanced_sampler(
        train_recs, cfg,
        hard_negative_topk_frac=cfg.HARD_NEGATIVE_TOPK_FRAC,
        motion_cache=motion_cache,
    )

    g_fold = torch.Generator(); g_fold.manual_seed(cfg.SEED + fold_idx)
    fold_train_loader = DataLoader(
        fold_train_ds, batch_size=cfg.BATCH_SIZE, sampler=fold_sampler,
        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
        worker_init_fn=seed_worker, generator=g_fold,
        persistent_workers=(cfg.NUM_WORKERS > 0),
    )
    # fix: pass g_fold through so this fold's val loader is seeded consistently with its
    # train loader, instead of silently falling back to the global `g`.
    fold_val_loader = make_eval_loader(fold_val_ds, cfg, generator=g_fold)

    fold_model = SuspiciousActivityModel(num_frames=cfg.NUM_FRAMES, dropout=0.3,
                                         pretrained=True).to(device)
    fold_optim   = torch.optim.AdamW(fold_model.parameters(), lr=cfg.LR,
                                      weight_decay=cfg.WEIGHT_DECAY)
    fold_crit    = FocalLoss(alpha=cfg.FOCAL_ALPHA, gamma=cfg.FOCAL_GAMMA)
    fold_scaler  = GradScaler(AMP_DEVICE_TYPE, enabled=AMP_ENABLED)
    fold_sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        fold_optim, mode="max", factor=0.5, patience=3
    )

    best_prauc  = -1.0
    patience    = 0
    best_val_metrics = None

    for epoch in range(1, n_epochs + 1):
        run_epoch(fold_model, fold_train_loader, fold_optim, fold_crit,
                  device, fold_scaler, train=True,
                  desc=f"Fold {fold_idx+1} Epoch {epoch}/{n_epochs} [train]")
        val_m = run_epoch(fold_model, fold_val_loader, fold_optim, fold_crit,
                          device, fold_scaler, train=False,
                          desc=f"Fold {fold_idx+1} Epoch {epoch}/{n_epochs} [val]")
        fold_sched.step(val_m["pr_auc"])

        if val_m["pr_auc"] > best_prauc:
            best_prauc        = val_m["pr_auc"]
            best_val_metrics  = val_m
            patience          = 0
        else:
            patience += 1
            if patience >= cfg.EARLY_STOP_PATIENCE:
                break

    print(f"  Fold {fold_idx+1} best val PR-AUC = {best_prauc:.4f}")
    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return best_val_metrics


### Optional 3-fold cross-validation

Runs the expensive robustness check only when enabled, training three folds for mean and standard-deviation metrics.

In [34]:
# ── Optional 3-fold CV on the train+validation pool ──────────────────────────
# Runtime note: full CV retrains 3 extra models and can easily exceed a 10-hour Kaggle GPU session.
# It is OFF by default. Set CFG.RUN_3FOLD_CV = True in the config cell only when you have enough GPU time.

cv_prauc_list = []
cv_mean = None
cv_std = None

if getattr(CFG, "RUN_3FOLD_CV", True):
    print("Starting 3-fold cross-validation (video-level stratified) ...")
    print(f"Note: using {getattr(CFG, 'CV_EPOCHS', 5)}-epoch budget per fold for speed.\n")

    # CV must never touch test_records: restricted to train_records + val_records only, so the
    # held-out test set used in Sections 10-13 is never leaked into CV-time training.
    cv_pool       = train_records + val_records
    skf           = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.SEED)
    # Stratify CV by class_name, not only binary label, to keep Burglary/Fighting/Stealing/Normal balanced across folds.
    all_labels    = [r["class_name"] for r in cv_pool]

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(cv_pool, all_labels)):
        tr_recs = [cv_pool[i] for i in tr_idx]
        va_recs = [cv_pool[i] for i in va_idx]

        fold_metrics = run_cv_fold(
            fold_idx, tr_recs, va_recs, CFG, DEVICE,
            motion_cache=motion_cache, n_epochs=getattr(CFG, "CV_EPOCHS", 5)
        )
        cv_prauc_list.append(fold_metrics["pr_auc"] if fold_metrics is not None else float("nan"))

    cv_mean = float(np.nanmean(cv_prauc_list))
    cv_std  = float(np.nanstd(cv_prauc_list))
    print(f"\n3-Fold CV PR-AUC: {cv_mean:.3f} ± {cv_std:.3f}")
else:
    print("3-fold CV skipped by default to keep the full notebook within the GPU time limit.")
    print("Set CFG.RUN_3FOLD_CV = True only when you have enough extra GPU time.")


Starting 3-fold cross-validation (video-level stratified) ...
Note: using 5-epoch budget per fold for speed.

Sampler class counts: Counter({'Normal': 142, 'Burglary': 57, 'Stealing': 56, 'Fighting': 28})
Expected sampled clips/epoch by class: {'Burglary': 91.0, 'Fighting': 91.0, 'Normal': 293.1, 'Stealing': 91.0}


  Fold 1 best val PR-AUC = 0.8772
Sampler class counts: Counter({'Normal': 142, 'Stealing': 57, 'Burglary': 56, 'Fighting': 28})
Expected sampled clips/epoch by class: {'Burglary': 91.0, 'Fighting': 91.0, 'Normal': 293.1, 'Stealing': 91.0}


  Fold 2 best val PR-AUC = 0.8822
Sampler class counts: Counter({'Normal': 142, 'Stealing': 57, 'Burglary': 57, 'Fighting': 28})
Expected sampled clips/epoch by class: {'Burglary': 91.3, 'Fighting': 91.3, 'Normal': 294.1, 'Stealing': 91.3}


  Fold 3 best val PR-AUC = 0.8520

3-Fold CV PR-AUC: 0.870 ± 0.013


### Final summary export helpers

Defines helper functions for optional variables and JSON-safe conversion, then builds the final experiment summary dictionary.

In [35]:
def optional_var(name, default=None):
    """Returns a variable if the corresponding cell was run; otherwise returns default."""
    return globals().get(name, default)


def to_jsonable(obj):
    """Converts NumPy scalar/array objects to normal Python types before JSON export."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


final_summary = {
    "model"                     : "MobileNetV3-Large + TSM",
    "num_frames"                : CFG.NUM_FRAMES,
    "img_size"                  : CFG.IMG_SIZE,
    "train_samples_per_video"   : int(getattr(CFG, "TRAIN_SAMPLES_PER_VIDEO", 1)),
    "best_val_pr_auc_clip_level": ckpt_manager.best_pr_auc,
    "best_val_roc_auc_clip_level": ckpt_manager.best_roc_auc,
    "selected_threshold_clip_f1" : best_thresh,
    "test_metrics_default_threshold_clip_level" : test_metrics_default,
    "test_metrics_tuned_threshold_clip_level"   : test_metrics_tuned,
    "val_video_level_metrics"   : {
        "max": val_video_max_metrics,
        "top3_mean": val_video_top3_metrics,
    },
    "test_video_level_metrics"  : {
        "max": test_video_max_metrics,
        "top3_mean": test_video_top3_metrics,
    },
    "event_level_metrics_test"  : event_metrics,
    "alert_logic_params"        : best_alert_params,
    "alert_logic_tuning_note"   : (
        "alpha/threshold/N are selected on validation sliding-window predictions and then applied once "
        "to the held-out test sliding-window predictions. This avoids test-set alert tuning leakage."
    ),
    "calibration_ece_clip_level": ece,
    "deployment_benchmark_cpu"  : bench,
    "tsm_ablation"              : ablation_df.to_dict(orient="records"),
    "cv_3fold"                  : {"pr_auc_mean": optional_var("cv_mean"), "pr_auc_std": optional_var("cv_std"),
                                   "per_fold": optional_var("cv_prauc_list", [])},
}

with open(os.path.join(CFG.OUTPUT_DIR, "final_summary.json"), "w") as f:
    json.dump(to_jsonable(final_summary), f, indent=2)

print(json.dumps(to_jsonable(final_summary), indent=2))
print(f"\nAll artifacts saved to: {CFG.OUTPUT_DIR}")
print("Files:", os.listdir(CFG.OUTPUT_DIR))


{
  "model": "MobileNetV3-Large + TSM",
  "num_frames": 8,
  "img_size": 160,
  "train_samples_per_video": 2,
  "best_val_pr_auc_clip_level": 0.8459807917601359,
  "best_val_roc_auc_clip_level": 0.8634423897581792,
  "selected_threshold_clip_f1": 0.490234375,
  "test_metrics_default_threshold_clip_level": {
    "pr_auc": 0.9380221611900564,
    "roc_auc": 0.922475106685633,
    "f1": 0.88,
    "recall": 0.868421052631579,
    "precision": 0.8918918918918919,
    "mse": 0.11351216858873765,
    "rmse": 0.3369156698474229
  },
  "test_metrics_tuned_threshold_clip_level": {
    "pr_auc": 0.9380221611900564,
    "roc_auc": 0.922475106685633,
    "f1": 0.868421052631579,
    "recall": 0.868421052631579,
    "precision": 0.868421052631579,
    "mse": 0.11351216858873765,
    "rmse": 0.3369156698474229
  },
  "val_video_level_metrics": {
    "max": {
      "pr_auc": 0.8603596462363055,
      "score_col": "score_max",
      "n_videos": 75,
      "roc_auc": 0.8438833570412518
    },
    "top3_m

### Expanded-manifest diagnostic check

Confirms that the notebook is using the expanded 500-video manifest and prints the current class and binary-label counts for audit purposes.

In [36]:
import os
import pandas as pd
from collections import Counter

# Diagnostic check: confirm the notebook is still using the expanded manifest, not the old 300-video folder scan.
if CFG.MANIFEST_CSV is not None and os.path.exists(CFG.MANIFEST_CSV):
    manifest_check_df = pd.read_csv(CFG.MANIFEST_CSV)
    print("Manifest file:", CFG.MANIFEST_CSV)
    print("\nManifest class count:")
    print(manifest_check_df["class_name"].value_counts())
    print("\nManifest binary label count:")
    print(manifest_check_df["label"].value_counts())
else:
    print("CFG.MANIFEST_CSV is not set or not found.")
    print("Current in-memory manifest class count:")
    print(Counter([r["class_name"] for r in manifest]))


Manifest file: /kaggle/working/expanded_normal_manifest.csv

Manifest class count:
class_name
Normal      250
Burglary    100
Stealing    100
Fighting     50
Name: count, dtype: int64

Manifest binary label count:
label
1    250
0    250
Name: count, dtype: int64


### Failure-case audit

Identifies videos and classes that still confuse the model so errors can be inspected and reported clearly.

In [37]:

# ── Failure-case audit: shows the data types still confusing the model ─────────
# Use this to decide what data to add next. It prints highest-scoring Normal videos
# and lowest-scoring Suspicious videos using the full-video score, not one random clip.

score_path = os.path.join(CFG.OUTPUT_DIR, "test_video_level_scores.csv")
print("Reading:", score_path)
if not os.path.exists(score_path):
    raise FileNotFoundError(
        f"{score_path} not found. Run the sliding-window evaluation cell before this failure-case audit."
    )
video_audit_df = pd.read_csv(score_path)

print("\nTop Normal videos by max suspicious score — likely false-alarm risk:")
print(
    video_audit_df[video_audit_df.true_label == 0]
    .sort_values("score_max", ascending=False)
    .head(10)[["path", "class_name", "score_max", "score_topk_mean", "n_windows"]]
    .to_string(index=False)
)

print("\nLowest Suspicious videos by max suspicious score — likely missed-event risk:")
print(
    video_audit_df[video_audit_df.true_label == 1]
    .sort_values("score_max", ascending=True)
    .head(10)[["path", "class_name", "score_max", "score_topk_mean", "n_windows"]]
    .to_string(index=False)
)


Reading: /kaggle/working/checkpoints_expanded_normal_mnv3_large_confirmed/test_video_level_scores.csv

Top Normal videos by max suspicious score — likely false-alarm risk:
                                                                                                                                                      path class_name  score_max  score_topk_mean  n_windows
/kaggle/input/datasets/vigneshwar472/ucaucf-crime-annotation-dataset/UCF_Crimes/UCF_Crimes/Videos/Training_Normal_Videos_Anomaly/Normal_Videos484_x264.mp4     Normal   0.994141         0.993978        200
/kaggle/input/datasets/vigneshwar472/ucaucf-crime-annotation-dataset/UCF_Crimes/UCF_Crimes/Videos/Training_Normal_Videos_Anomaly/Normal_Videos320_x264.mp4     Normal   0.844238         0.842285         32
/kaggle/input/datasets/vigneshwar472/ucaucf-crime-annotation-dataset/UCF_Crimes/UCF_Crimes/Videos/Training_Normal_Videos_Anomaly/Normal_Videos709_x264.mp4     Normal   0.716797         0.648763        135
/kaggle/

### Data-source audit

Checks final train/validation/test records to confirm no Testing_Normal leakage and to summarize dataset sources.

In [38]:

# ── Data-source audit ────────────────────────────────────────────────────────
# Confirms that the expanded manifest is being used and that official testing-normal
# videos did not leak into the train/validation/test pool.

manifest_audit = pd.read_csv(CFG.MANIFEST_CSV)
print("Manifest:", CFG.MANIFEST_CSV)
print("\nClass counts:")
print(manifest_audit["class_name"].value_counts())
print("\nBinary label counts:")
print(manifest_audit["label"].value_counts())

n_testing_normal = manifest_audit["video_path"].str.contains(
    "Testing_Normal_Videos_Anomaly", regex=False
).sum()
print("\nTesting_Normal_Videos_Anomaly files in manifest:", int(n_testing_normal))
assert n_testing_normal == 0, "Testing_Normal_Videos_Anomaly leakage detected."
print("✓ Data-source audit passed.")


Manifest: /kaggle/working/expanded_normal_manifest.csv

Class counts:
class_name
Normal      250
Burglary    100
Stealing    100
Fighting     50
Name: count, dtype: int64

Binary label counts:
label
1    250
0    250
Name: count, dtype: int64

Testing_Normal_Videos_Anomaly files in manifest: 0
✓ Data-source audit passed.
